# Subthreshold Analysis Notebook

This notebook extracts subthreshold activity from Spike Detector analyzed sessions and analyzes its relation to SS/CS firing and event timing.

Pipeline:
1. Load analyzed sessions from the same master folder/glob pattern used by the analysis workflow.
2. Bleaching correction with sliding-window median baseline (default 1000 ms).
3. Mask and interpolate SS/CS spike windows on raw traces.
4. Zero-phase low-pass filter (default 80 Hz, configurable 50-100 Hz).
5. Analyze subthreshold vs SS/CS spikes and vs event timing (manual or auto-detected puff events).

In [ ]:
# ==============================
# SETTINGS (all user settings)
# ==============================
DATA_FOLDER = r'/Volumes/T7 Shield/20250509_5PCs_nopuff'
SESSION_GLOB = 'spike_detection/*_analyzed.npz'
STRICT_CELL_NAME_MATCH = True

# Preprocessing (explicit pipeline options)
PREPROCESS_CONFIG = {
    'bleach_median_window_ms': 1000.0,
    'spike_mask_ss_ms': (-2.0, 3.0),
    'spike_mask_cs_ms': (-8.0, 16.0),
    'interpolation_kind': 'linear',
    'second_median_enabled': True,
    'second_median_window_ms': 10.0,
    'lowpass_enabled': True,
    'lowpass_cutoff_hz': 100.0,
    'lowpass_order': 2,
}

# Backward-compatible aliases used throughout notebook cells
BLEACH_MEDIAN_WINDOW_MS = float(PREPROCESS_CONFIG['bleach_median_window_ms'])
SPIKE_MASK_SS_MS = tuple(PREPROCESS_CONFIG['spike_mask_ss_ms'])
SPIKE_MASK_CS_MS = tuple(PREPROCESS_CONFIG['spike_mask_cs_ms'])
INTERPOLATION_KIND = str(PREPROCESS_CONFIG['interpolation_kind'])
SECOND_MEDIAN_ENABLED = bool(PREPROCESS_CONFIG['second_median_enabled'])
SECOND_MEDIAN_WINDOW_MS = float(PREPROCESS_CONFIG['second_median_window_ms'])
LOWPASS_ENABLED = bool(PREPROCESS_CONFIG['lowpass_enabled'])
LOWPASS_CUTOFF_HZ = float(PREPROCESS_CONFIG['lowpass_cutoff_hz'])
LOWPASS_ORDER = int(PREPROCESS_CONFIG['lowpass_order'])

# Coherence / MSC settings for synchrony analyses
COHERENCE_CONFIG = {
    'enabled': True,
    'freq_band_hz': (0.2, 100.0),
    'nperseg': 1024,
    'noverlap_ratio': 0.5,
    'n_shuffle': 1000,
    'show_progress': False,  # inner shuffle progress floods notebook output for many pairs
    'progress_leave': False,
    'progress_mininterval_s': 0.2,
    'shuffle_mode': 'phase_randomized',
    'jitter_ms': 200.0,
    'min_shift_ms': 100.0,
    'rng_seed': 123,
    'max_plot_freq_hz': 100.0,
    'plot_log_x': False,
    'plot_y_max': 0.5,
    'max_plot_pairs': 12,
    'max_signal_samples': 150000,
    'max_fs_hz': 300.0,
    'show_pair_spectra': True,
}

# SS spike-train MSC settings (IFR via Gaussian smoothing)
SS_MSC_CONFIG = {
    'enabled': True,
    'ifreq_sigma_ms': 5,
    'scale_to_hz': True,
    'mask_post_event_period': True,
    'freq_band_hz': (0.2, 100.0),
    'nperseg': 1024,
    'noverlap_ratio': 0.5,
    'n_shuffle': 1000,
    'show_progress': False,  # inner shuffle progress floods notebook output for many pairs
    'progress_leave': False,
    'progress_mininterval_s': 0.2,
    'shuffle_mode': 'circular_shift',
    'jitter_ms': 200.0,
    'min_shift_ms': 100.0,
    'rng_seed': 123,
    'max_plot_freq_hz': 100.0,
    'plot_log_x': False,
    'plot_y_max': 0.1,
    'max_plot_pairs': 12,
    'max_signal_samples': 150000,
    'max_fs_hz': 300.0,
    'show_pair_spectra': True,
}

# Example visualization selection
EXAMPLE_SESSION_IDX = 0
EXAMPLE_CELL_IDX = 0
EXAMPLE_T_RANGE_MS = (1800.0, 3200.0)
EXAMPLE_EQUAL_Y_SPAN = True
EXAMPLE_CENTER_EACH_TRACE = True

# Whole-recording subthreshold synchrony (pooled across sessions)
SUBTH_SYNC_CONFIG = {
    'enabled': True,
    'metric': 'pearson',
    'null_mode': 'circular_shift',
    'jitter_ms': 200.0,
    'min_shift_ms': 100.0,
    'n_null_shuffles': 1000,
    'show_progress': False,  # render compact final tables/matrices instead
    'progress_leave': False,
    'progress_mininterval_s': 0.2,
    'rng_seed': 123,
    'alpha': 0.05,
    'min_valid_samples': 50,
    'show_per_session_matrices': False,
    'mask_post_event_period': True,
}

# Shared typography for every correlation/synchrony matrix in this notebook.
SUBTH_CORR_MATRIX_STYLE = {
    'value_fontsize': 10,
    'stat_fontsize': 7,
    'tick_label_fontsize': 9,
    'title_fontsize': 12,
    'colorbar_label_fontsize': 9,
    'colorbar_tick_fontsize': 8,
    'pair_annot_fontsize': 9,
    'value_fontweight': 'bold',
}

# Spike-locked analysis settings
STA_WINDOW_SS_MS = (-50.0, 50.0)
STA_WINDOW_CS_MS = (-200.0, 200.0)
STA_SHOW_MASK_OVERLAY = True
STA_MASK_OVERLAY_COLOR = '#9CA3AF'
STA_MASK_OVERLAY_ALPHA = 0.2
MIN_SPIKES_FOR_STA = 10

# Event settings
EVENT_ONSET_MS = 2510.0
EVENT_DETECTION_CONFIG = {
    'auto_detect_event': False,
    'behavior_csv_glob': '**/*.csv',
    'behavior_state_col': 'Arduino_State',
    'behavior_time_col_candidates': ['Time_s', 'Time_ms', 'time_s', 'time_ms'],
    'behavior_state_on_value': 1,
    'behavior_time_units': 's',
    'max_events_per_session': None,
    'verbose': True,
}
EVENT_LOCK_WINDOW_MS = (-600.0, 1200.0)
EVENT_BASELINE_MS = (-300.0, -0.0)
EVENT_RESPONSE_MS = (0.0, 300.0)

# Optional export
EXPORT_RESULTS = False
EXPORT_DIR = f'{DATA_FOLDER}/subthreshold_exports'

In [ ]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.ndimage import median_filter, gaussian_filter1d
from scipy.signal import butter, sosfiltfilt, coherence, decimate

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from IPython.display import display, clear_output
except Exception:
    def display(obj):
        print(obj)
    def clear_output(wait=False):
        return None

try:
    import ipywidgets as widgets
    HAS_IPYWIDGETS = True
except Exception:
    HAS_IPYWIDGETS = False

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)

NOTEBOOK_OUTPUT_CONFIG = {
    'table_top_n': 12,
    'pair_plot_limit': 6,
    'clear_intermediate_progress': True,
}

def display_top(df, n=None, label=None, sort_by=None, ascending=False):
    """Display a bounded, informative table without flooding notebook outputs."""
    n = int(NOTEBOOK_OUTPUT_CONFIG['table_top_n'] if n is None else n)
    view = df.sort_values(sort_by, ascending=ascending) if sort_by is not None and len(df) else df
    if label:
        print(f'{label}: showing {min(len(view), n)} of {len(view)} rows')
    display(view.head(n))

sns.set_theme(style='white', context='talk')
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'axes.grid': False,
    'figure.dpi': 120,
})

SUBTH_COLORS = {
    'raw': '#4C72B0',
    'baseline': '#DD8452',
    'corrected': '#4C72B0',
    'despiked': '#55A868',
    'subthreshold': '#8172B2',
    'ss': '#4C72B0',
    'cs': '#C44E52',
    'event': '#55A868',
    'pooled': '#4C72B0',
}

if EXPORT_RESULTS:
    os.makedirs(EXPORT_DIR, exist_ok=True)

In [ ]:
# ==============================
# Session loading / validation
# ==============================
REQUIRED_KEYS = [
    'time_ms', 'raw_data', 'spike_times_cs', 'spike_times_ss',
    'cell_names', 'fs'
]

def _to_1d_float_array(x):
    arr = np.asarray(x, dtype=float).reshape(-1)
    return arr[np.isfinite(arr)]

def _validate_and_normalize_session(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    missing = [k for k in REQUIRED_KEYS if k not in d.files]
    if missing:
        raise ValueError(f'Missing keys: {missing}')

    time_ms = _to_1d_float_array(d['time_ms'])
    raw_data = np.asarray(d['raw_data'], dtype=float)
    if raw_data.ndim != 2:
        raise ValueError('raw_data must be 2D [samples, cells]')

    n_samples, n_cells = raw_data.shape
    if len(time_ms) != n_samples:
        raise ValueError('time_ms length mismatch')

    cell_names = [str(c) for c in np.asarray(d['cell_names']).tolist()]
    if len(cell_names) != n_cells:
        raise ValueError('cell_names length mismatch')

    spike_times_ss_raw = np.asarray(d['spike_times_ss'], dtype=object)
    spike_times_cs_raw = np.asarray(d['spike_times_cs'], dtype=object)
    if len(spike_times_ss_raw) != n_cells or len(spike_times_cs_raw) != n_cells:
        raise ValueError('spike_times arrays must be length n_cells')

    spike_times_ss = [_to_1d_float_array(spike_times_ss_raw[i]) for i in range(n_cells)]
    spike_times_cs = [_to_1d_float_array(spike_times_cs_raw[i]) for i in range(n_cells)]

    t0, t1 = float(time_ms[0]), float(time_ms[-1])
    for i in range(n_cells):
        spike_times_ss[i] = spike_times_ss[i][(spike_times_ss[i] >= t0) & (spike_times_ss[i] <= t1)]
        spike_times_cs[i] = spike_times_cs[i][(spike_times_cs[i] >= t0) & (spike_times_cs[i] <= t1)]

    fname = os.path.splitext(os.path.basename(npz_path))[0]
    session_name = fname.replace('_analyzed', '')

    return {
        'session_name': session_name,
        'npz_path': npz_path,
        'time_ms': time_ms,
        'raw_data': raw_data,
        'spike_times_ss': spike_times_ss,
        'spike_times_cs': spike_times_cs,
        'n_cells': n_cells,
        'cell_names': cell_names,
        'fs': float(d['fs']),
    }

def _find_session_paths(data_folder, preferred_glob):
    globs_to_try = [
        preferred_glob,
        'spike_detection/*_analyzed.npz',
        '*/*_analyzed.npz',
        '**/*_analyzed.npz',
    ]
    seen = set()
    out = []
    for g in globs_to_try:
        for p in sorted(glob.glob(os.path.join(data_folder, g), recursive=True)):
            if p not in seen:
                seen.add(p)
                out.append(p)
        if out:
            return out, g
    return [], preferred_glob

def load_sessions(data_folder, session_glob='spike_detection/*_analyzed.npz', strict_cell_name_match=True):
    paths, matched_glob = _find_session_paths(data_folder, session_glob)
    if not paths:
        raise FileNotFoundError(f'No session files found under {data_folder}; tried glob={session_glob}')

    loaded, excluded = [], []
    ref_n_cells, ref_cell_names = None, None

    for p in paths:
        try:
            s = _validate_and_normalize_session(p)
            if ref_n_cells is None:
                ref_n_cells = s['n_cells']
                ref_cell_names = s['cell_names']
            else:
                if s['n_cells'] != ref_n_cells:
                    raise ValueError('n_cells mismatch across sessions')
                if strict_cell_name_match and s['cell_names'] != ref_cell_names:
                    raise ValueError('cell_names mismatch (strict mode)')
            loaded.append(s)
        except Exception as e:
            excluded.append({'path': p, 'reason': str(e)})

    if not loaded:
        raise RuntimeError('All session files failed validation.')

    return loaded, excluded, matched_glob

In [ ]:
sessions, excluded_sessions, matched_glob = load_sessions(
    DATA_FOLDER,
    session_glob=SESSION_GLOB,
    strict_cell_name_match=STRICT_CELL_NAME_MATCH,
)

print(f'Loaded sessions: {len(sessions)}')
print(f'Excluded sessions: {len(excluded_sessions)}')
print(f'Matched glob: {matched_glob}')
if excluded_sessions:
    for ex in excluded_sessions:
        print(f'- {ex["path"]}: {ex["reason"]}')

n_cells = sessions[0]['n_cells']
cell_names = sessions[0]['cell_names']
summary_df = pd.DataFrame([{
    'session': s['session_name'],
    'duration_ms': float(s['time_ms'][-1] - s['time_ms'][0]),
    'n_cells': s['n_cells'],
    'fs': s['fs'],
    'ss_spikes_total': int(sum(len(x) for x in s['spike_times_ss'])),
    'cs_spikes_total': int(sum(len(x) for x in s['spike_times_cs'])),
} for s in sessions])
display_top(summary_df, label='Loaded-session summary')

In [ ]:
# ==============================
# Event auto-detection helpers
# ==============================
def _find_behavior_csv_for_session(session, event_cfg):
    if not bool(event_cfg.get('auto_detect_event', False)):
        return None

    root = Path(DATA_FOLDER)
    csv_glob = str(event_cfg.get('behavior_csv_glob', '**/*.csv'))
    candidates = sorted(root.glob(csv_glob))
    if not candidates:
        return None

    sname = str(session.get('session_name', '')).lower()
    for p in candidates:
        stem = p.stem.lower()
        if sname and (sname in stem or stem in sname):
            return str(p)

    # fallback: same parent folder if available
    npz_path = Path(str(session.get('npz_path', '')))
    for p in candidates:
        if npz_path.parent in p.parents:
            return str(p)

    return str(candidates[0])

def _detect_event_times_from_behavior_csv(csv_path, event_cfg):
    if csv_path is None or not os.path.exists(csv_path):
        return np.array([], dtype=float)

    df = pd.read_csv(csv_path)
    if df.empty:
        return np.array([], dtype=float)

    state_col = event_cfg.get('behavior_state_col', 'Arduino_State')
    cols_lower = {c.lower(): c for c in df.columns}
    state_col_real = cols_lower.get(str(state_col).lower(), None)
    if state_col_real is None:
        return np.array([], dtype=float)

    time_candidates = event_cfg.get('behavior_time_col_candidates', ['Time_s', 'Time_ms'])
    time_col_real = None
    for tc in time_candidates:
        if str(tc).lower() in cols_lower:
            time_col_real = cols_lower[str(tc).lower()]
            break
    if time_col_real is None:
        return np.array([], dtype=float)

    state_on = float(event_cfg.get('behavior_state_on_value', 1))
    state = pd.to_numeric(df[state_col_real], errors='coerce').fillna(0.0).values
    t_raw = pd.to_numeric(df[time_col_real], errors='coerce').values

    valid = np.isfinite(t_raw)
    if not np.any(valid):
        return np.array([], dtype=float)
    state = state[valid]
    t_raw = t_raw[valid]

    on_mask = state == state_on
    prev_on = np.r_[False, on_mask[:-1]]
    rising = on_mask & (~prev_on)
    ev = t_raw[rising]

    units = str(event_cfg.get('behavior_time_units', 's')).lower()
    if units in {'s', 'sec', 'second', 'seconds'}:
        ev_ms = ev * 1000.0
    else:
        ev_ms = ev

    max_n = event_cfg.get('max_events_per_session', None)
    ev_ms = np.asarray(ev_ms, dtype=float)
    ev_ms = np.unique(np.round(ev_ms, 3))
    if isinstance(max_n, (int, np.integer)) and max_n > 0:
        ev_ms = ev_ms[:int(max_n)]

    return ev_ms

def _get_session_event_times_ms(session, event_cfg, cache=None):
    cache = {} if cache is None else cache
    key = str(session.get('npz_path', session.get('session_name', 'unknown')))
    if key in cache:
        return cache[key]

    default_ev = np.array([float(event_cfg.get('event_time_ms', EVENT_ONSET_MS))], dtype=float)
    if not bool(event_cfg.get('auto_detect_event', False)):
        cache[key] = default_ev
        return default_ev

    csv_path = _find_behavior_csv_for_session(session, event_cfg)
    ev = _detect_event_times_from_behavior_csv(csv_path, event_cfg)
    if ev.size == 0:
        ev = default_ev

    cache[key] = np.asarray(ev, dtype=float)
    return cache[key]

EVENT_CONFIG = {
    'event_time_ms': EVENT_ONSET_MS,
    **EVENT_DETECTION_CONFIG,
}

In [ ]:
# ==============================
# Subthreshold extraction
# ==============================
def _ensure_odd(n):
    n = int(max(3, n))
    return n if n % 2 == 1 else n + 1

def sliding_median_baseline(trace, fs, window_ms=1000.0):
    win = _ensure_odd(int(round((window_ms / 1000.0) * fs)))
    baseline = median_filter(np.asarray(trace, dtype=float), size=win, mode='nearest')
    return baseline

def _mask_windows_from_spikes(n_samples, fs, spike_times_ms, window_ms):
    mask = np.zeros(n_samples, dtype=bool)
    if spike_times_ms is None:
        return mask
    sidx = np.round(np.asarray(spike_times_ms, dtype=float) * fs / 1000.0).astype(int)
    w0, w1 = float(window_ms[0]), float(window_ms[1])
    for p in sidx:
        a = int(round(p + (w0 / 1000.0) * fs))
        b = int(round(p + (w1 / 1000.0) * fs))
        lo, hi = sorted((a, b))
        lo = max(0, lo)
        hi = min(n_samples - 1, hi)
        if hi >= lo:
            mask[lo:hi+1] = True
    return mask

def mask_and_interpolate_spikes(trace, fs, ss_times_ms, cs_times_ms, ss_window_ms, cs_window_ms):
    x = np.asarray(trace, dtype=float).copy()
    n = len(x)
    bad = _mask_windows_from_spikes(n, fs, ss_times_ms, ss_window_ms) | _mask_windows_from_spikes(n, fs, cs_times_ms, cs_window_ms)
    if not np.any(bad):
        return x, bad

    good_idx = np.where(~bad)[0]
    if len(good_idx) < 2:
        return x, bad

    x[bad] = np.interp(np.where(bad)[0], good_idx, x[good_idx])
    return x, bad

def lowpass_zero_phase(trace, fs, cutoff_hz=80.0, order=3):
    x = np.asarray(trace, dtype=float)
    nyq = 0.5 * float(fs)
    c = float(cutoff_hz)
    if c <= 0 or c >= nyq:
        raise ValueError(f'Invalid cutoff_hz={c}; must be in (0, {nyq})')
    wn = c / nyq
    sos = butter(int(order), wn, btype='low', output='sos')
    return sosfiltfilt(sos, x)

def _sanitize_lowpass_cutoff_hz(fs, cutoff_hz):
    nyq = 0.5 * float(fs)
    c = float(cutoff_hz)
    if c <= 0:
        return max(1.0, 0.1 * nyq)
    if c >= nyq:
        return 0.95 * nyq
    return c

def _apply_optional_denoising(trace, fs, preprocess_cfg):
    out = np.asarray(trace, dtype=float)

    if bool(preprocess_cfg.get('second_median_enabled', False)):
        win_ms = float(preprocess_cfg.get('second_median_window_ms', 10.0))
        win_samples = _ensure_odd(int(round((win_ms / 1000.0) * float(fs))))
        out = median_filter(out, size=win_samples, mode='nearest')

    if bool(preprocess_cfg.get('lowpass_enabled', True)):
        cutoff_hz = _sanitize_lowpass_cutoff_hz(fs, preprocess_cfg.get('lowpass_cutoff_hz', 80.0))
        out = lowpass_zero_phase(
            out,
            fs,
            cutoff_hz=cutoff_hz,
            order=int(preprocess_cfg.get('lowpass_order', 3)),
        )

    return out

def build_subthreshold_dataset(sessions):
    out = []
    for s in sessions:
        raw = np.asarray(s['raw_data'], dtype=float)
        fs = float(s['fs'])
        n_samples, n_cells_local = raw.shape

        baseline_mat = np.zeros_like(raw, dtype=float)
        corrected_mat = np.zeros_like(raw, dtype=float)
        despiked_mat = np.zeros_like(raw, dtype=float)
        denoised_mat = np.zeros_like(raw, dtype=float)
        subth_mat = np.zeros_like(raw, dtype=float)
        masked_fraction = np.zeros(n_cells_local, dtype=float)

        for c in range(n_cells_local):
            trace = raw[:, c]
            baseline = sliding_median_baseline(trace, fs, window_ms=BLEACH_MEDIAN_WINDOW_MS)
            corrected = trace - baseline
            despiked, bad = mask_and_interpolate_spikes(
                corrected, fs,
                s['spike_times_ss'][c], s['spike_times_cs'][c],
                SPIKE_MASK_SS_MS, SPIKE_MASK_CS_MS
            )
            denoised = _apply_optional_denoising(despiked, fs, PREPROCESS_CONFIG)

            baseline_mat[:, c] = baseline
            corrected_mat[:, c] = corrected
            despiked_mat[:, c] = despiked
            denoised_mat[:, c] = denoised
            subth_mat[:, c] = denoised
            masked_fraction[c] = float(np.mean(bad))

        s_out = dict(s)
        s_out.update({
            'baseline': baseline_mat,
            'corrected': corrected_mat,
            'despiked': despiked_mat,
            'denoised': denoised_mat,
            'subthreshold': subth_mat,
            'masked_fraction': masked_fraction,
        })
        out.append(s_out)

    return out

In [ ]:
processed_sessions = build_subthreshold_dataset(sessions)

qc_df = pd.DataFrame([{
    'session': s['session_name'],
    'n_cells': s['n_cells'],
    'mean_masked_fraction': float(np.mean(s['masked_fraction'])),
    'max_masked_fraction': float(np.max(s['masked_fraction'])),
} for s in processed_sessions])
display_top(qc_df, label='Preprocessing QC summary')

In [ ]:
# Example subthreshold visualization with interactive window controls
def plot_example_window(session_idx, cell_idx, center_ms, window_ms):
    s = processed_sessions[int(np.clip(session_idx, 0, len(processed_sessions) - 1))]
    c = int(np.clip(cell_idx, 0, s['n_cells'] - 1))
    t = s['time_ms']
    half = max(50.0, float(window_ms) / 2.0)
    t0 = float(center_ms) - half
    t1 = float(center_ms) + half
    mask = (t >= t0) & (t <= t1)

    if np.sum(mask) < 5:
        print('Selected window has too few samples; increase window size or move center.')
        return

    fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True, constrained_layout=True)
    axes[0].plot(t[mask], s['raw_data'][mask, c], color=SUBTH_COLORS['raw'], lw=1)
    axes[0].set_title(f"Raw trace | session={s['session_name']} | cell={s['cell_names'][c]}")

    axes[1].plot(t[mask], s['baseline'][mask, c], color=SUBTH_COLORS['baseline'], lw=1.2)
    axes[1].set_title('Sliding median baseline (bleaching estimate)')

    axes[2].plot(t[mask], s['corrected'][mask, c], color=SUBTH_COLORS['corrected'], lw=1)
    axes[2].set_title('Bleaching-corrected trace (raw - baseline)')

    axes[3].plot(t[mask], s['despiked'][mask, c], color=SUBTH_COLORS['despiked'], lw=1)
    axes[3].set_title('After SS/CS masking + interpolation')

    axes[4].plot(t[mask], s['subthreshold'][mask, c], color=SUBTH_COLORS['subthreshold'], lw=1.4)
    axes[4].set_title(f'Subthreshold (zero-phase low-pass {LOWPASS_CUTOFF_HZ:g} Hz)')
    axes[4].set_xlabel('Time (ms)')

    if bool(globals().get('EXAMPLE_EQUAL_Y_SPAN', True)):
        trace_list = [
            np.asarray(s['raw_data'][mask, c], dtype=float),
            np.asarray(s['baseline'][mask, c], dtype=float),
            np.asarray(s['corrected'][mask, c], dtype=float),
            np.asarray(s['despiked'][mask, c], dtype=float),
            np.asarray(s['subthreshold'][mask, c], dtype=float),
        ]
        spans = []
        for tr in trace_list:
            finite = tr[np.isfinite(tr)]
            if finite.size:
                spans.append(float(np.nanmax(finite) - np.nanmin(finite)))
        if spans:
            common_span = max(1e-9, max(spans)) * 1.05
            if bool(globals().get('EXAMPLE_CENTER_EACH_TRACE', True)):
                for ax, tr in zip(axes, trace_list):
                    finite = tr[np.isfinite(tr)]
                    if finite.size == 0:
                        continue
                    mid = float(np.nanmedian(finite))
                    ax.set_ylim(mid - 0.5 * common_span, mid + 0.5 * common_span)
            else:
                global_min = min(float(np.nanmin(tr[np.isfinite(tr)])) for tr in trace_list if np.isfinite(tr).any())
                global_max = max(float(np.nanmax(tr[np.isfinite(tr)])) for tr in trace_list if np.isfinite(tr).any())
                if np.isfinite(global_min) and np.isfinite(global_max):
                    pad = 0.025 * max(1e-9, global_max - global_min)
                    for ax in axes:
                        ax.set_ylim(global_min - pad, global_max + pad)

    for ax in axes:
        sns.despine(ax=ax, left=True, bottom=True)

    plt.show()
    plt.close(fig)  # avoid a second automatic inline render and figure accumulation

default_center = float(np.mean(EXAMPLE_T_RANGE_MS))
default_window = float(EXAMPLE_T_RANGE_MS[1] - EXAMPLE_T_RANGE_MS[0])

if HAS_IPYWIDGETS:
    t_min = float(min(s['time_ms'][0] for s in processed_sessions))
    t_max = float(max(s['time_ms'][-1] for s in processed_sessions))

    session_slider = widgets.IntSlider(
        value=int(EXAMPLE_SESSION_IDX), min=0, max=len(processed_sessions) - 1, step=1, description='Session', continuous_update=True
    )
    cell_slider = widgets.IntSlider(
        value=int(EXAMPLE_CELL_IDX), min=0, max=n_cells - 1, step=1, description='Cell', continuous_update=True
    )
    center_slider = widgets.FloatSlider(
        value=default_center, min=t_min, max=t_max, step=10.0, description='Center (ms)', continuous_update=True
    )
    window_slider = widgets.FloatSlider(
        value=max(100.0, default_window), min=100.0, max=min(10000.0, t_max - t_min), step=50.0,
        description='Window (ms)', continuous_update=True
    )

    ui = widgets.VBox([session_slider, cell_slider, center_slider, window_slider])
    out = widgets.Output()

    def _refresh_example_plot(change=None):
        out.clear_output(wait=True)
        with out:
            plot_example_window(session_slider.value, cell_slider.value, center_slider.value, window_slider.value)

    for control in (session_slider, cell_slider, center_slider, window_slider):
        control.observe(_refresh_example_plot, names='value')

    display(ui, out)
    _refresh_example_plot()
else:
    print('ipywidgets unavailable; showing static window. Install ipywidgets for sliders.')
    plot_example_window(EXAMPLE_SESSION_IDX, EXAMPLE_CELL_IDX, default_center, max(100.0, default_window))

In [ ]:
# ==============================
# Pooled subthreshold synchrony (whole recording)
# ==============================
def _subthsync_matrix_font(name, default):
    cfg = globals().get('SUBTH_CORR_MATRIX_STYLE', {})
    return float(cfg.get(name, default))


def _subthsync_safe_corr(x, y, min_valid_samples=50):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    good = np.isfinite(x) & np.isfinite(y)
    if np.sum(good) < int(min_valid_samples):
        return np.nan
    x = x[good]
    y = y[good]
    if np.std(x) <= 0 or np.std(y) <= 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def _subthsync_jitter_trace(trace, fs, jitter_ms, rng):
    trace = np.asarray(trace, dtype=float)
    n = trace.size
    if n == 0:
        return trace.copy()
    jitter_samples = max(1.0, float(jitter_ms) * float(fs) / 1000.0)
    x = np.arange(n, dtype=float)
    sample_points = np.clip(x + rng.uniform(-jitter_samples, jitter_samples, size=n), 0.0, float(max(0, n - 1)))
    return np.interp(sample_points, x, trace)


def _subthsync_circular_shift_trace(trace, fs, min_shift_ms, rng):
    trace = np.asarray(trace, dtype=float)
    n = trace.size
    if n < 2:
        return trace.copy()

    min_shift = max(1, int(round(float(min_shift_ms) * float(fs) / 1000.0)))
    if n <= 2 * min_shift:
        shift = int(rng.integers(1, n))
    else:
        shift = int(rng.integers(min_shift, n - min_shift))
    return np.roll(trace, shift)


def _subthsync_event_times_for_session(session_obj):
    if '_get_session_event_times_ms' in globals():
        try:
            return np.asarray(_get_session_event_times_ms(session_obj, EVENT_CONFIG, cache={}), dtype=float)
        except Exception:
            pass
    return np.array([float(EVENT_ONSET_MS)], dtype=float)


def _subthsync_mask_trace(trace, time_ms, event_times_ms, window_ms):
    out = np.asarray(trace, dtype=float).copy()
    t = np.asarray(time_ms, dtype=float)
    if out.size == 0 or t.size != out.size:
        return out
    w0, w1 = float(window_ms[0]), float(window_ms[1])
    bad = np.zeros(out.shape, dtype=bool)
    for event_time_ms in np.asarray(event_times_ms, dtype=float):
        bad |= (t >= (event_time_ms + w0)) & (t <= (event_time_ms + w1))
    out[bad] = np.nan
    return out


def _subthsync_shuffle_significance(trace_a, trace_b, fs, cfg, rng):
    min_valid = int(cfg.get('min_valid_samples', 50))
    trace_a = np.asarray(trace_a, dtype=float)
    trace_b = np.asarray(trace_b, dtype=float)
    good = np.isfinite(trace_a) & np.isfinite(trace_b)
    if np.sum(good) < min_valid:
        return np.nan, np.nan, np.nan

    clean_a = trace_a[good]
    clean_b = trace_b[good]
    obs = _subthsync_safe_corr(clean_a, clean_b, min_valid_samples=min_valid)
    if not np.isfinite(obs):
        return np.nan, np.nan, np.nan

    null_mode = str(cfg.get('null_mode', 'circular_shift')).lower()
    jitter_ms = float(cfg.get('jitter_ms', 20.0))
    min_shift_ms = float(cfg.get('min_shift_ms', 100.0))
    n_shuffles = int(cfg.get('n_null_shuffles', 1000))
    show_progress = bool(cfg.get('show_progress', True))
    progress_leave = bool(cfg.get('progress_leave', False))
    progress_mininterval = float(cfg.get('progress_mininterval_s', 0.2))
    null_vals = []
    for _ in tqdm(range(n_shuffles), desc='Pearson shuffles', disable=not show_progress, leave=progress_leave, mininterval=progress_mininterval):
        if null_mode in {'jitter', 'jitter_interp', 'interpolated_jitter'}:
            null_b = _subthsync_jitter_trace(clean_b, fs, jitter_ms, rng)
        else:
            null_b = _subthsync_circular_shift_trace(clean_b, fs, min_shift_ms, rng)
        r_null = _subthsync_safe_corr(clean_a, null_b, min_valid_samples=min_valid)
        if np.isfinite(r_null):
            null_vals.append(float(r_null))

    null_vals = np.asarray(null_vals, dtype=float)
    null_vals = null_vals[np.isfinite(null_vals)]
    if null_vals.size == 0:
        return float(obs), np.nan, np.nan

    p_two = (np.sum(np.abs(null_vals) >= abs(obs)) + 1.0) / (null_vals.size + 1.0)
    z = (obs - np.mean(null_vals)) / (np.std(null_vals) + 1e-9)
    return float(obs), float(p_two), float(z)


def _subthsync_prepare_coh_trace(x, fs, coh_cfg):
    x = np.asarray(x, dtype=float)
    fs_eff = float(fs)

    max_fs = float(coh_cfg.get('max_fs_hz', 250.0))
    if np.isfinite(max_fs) and max_fs > 0 and fs_eff > max_fs:
        q = int(np.ceil(fs_eff / max_fs))
        if q > 1 and x.size >= max(32, 8 * q):
            try:
                x = decimate(x, q, ftype='fir', zero_phase=True)
                fs_eff = fs_eff / float(q)
            except Exception:
                x = x[::q]
                fs_eff = fs_eff / float(q)

    max_n = int(coh_cfg.get('max_signal_samples', 120000))
    if max_n > 0 and x.size > max_n:
        # Keep sampling rate unchanged so the Nyquist limit stays high enough
        # to display the requested frequency band (e.g., up to 100 Hz).
        x = x[:max_n]

    return x, fs_eff


def _subthsync_phase_randomize(x, rng):
    x = np.asarray(x, dtype=float)
    n = x.size
    if n < 8:
        return x.copy()

    x0 = x - np.mean(x)
    spec = np.fft.rfft(x0)
    if spec.size <= 2:
        return x.copy()

    rand_phase = rng.uniform(0.0, 2.0 * np.pi, size=spec.size)
    rand_phase[0] = 0.0
    if n % 2 == 0:
        rand_phase[-1] = 0.0

    mag = np.abs(spec)
    new_spec = mag * np.exp(1j * rand_phase)
    y = np.fft.irfft(new_spec, n=n)
    return y + np.mean(x)


def _subthsync_coh_null_surrogate(y, fs, coh_cfg, rng):
    y = np.asarray(y, dtype=float)
    n = y.size
    if n < 2:
        return y.copy()

    mode = str(coh_cfg.get('shuffle_mode', 'phase_randomized')).lower()
    if mode == 'jitter':
        return _subthsync_jitter_trace(y, fs, float(coh_cfg.get('jitter_ms', 200.0)), rng)
    if mode == 'circular_shift':
        min_shift_ms = float(coh_cfg.get('min_shift_ms', 100.0))
        min_shift = max(1, int(round(min_shift_ms * float(fs) / 1000.0)))
        if n <= 2 * min_shift:
            shift = int(rng.integers(1, n))
        else:
            shift = int(rng.integers(min_shift, n - min_shift))
        return np.roll(y, shift)

    return _subthsync_phase_randomize(y, rng)


def _subthsync_coherence_with_shuffle(trace_a, trace_b, fs, coh_cfg, rng):
    if not bool(coh_cfg.get('enabled', True)):
        return None

    x = np.asarray(trace_a, dtype=float)
    y = np.asarray(trace_b, dtype=float)
    good = np.isfinite(x) & np.isfinite(y)
    if np.sum(good) < 10:
        return None

    x = x[good]
    y = y[good]
    x, fs_eff = _subthsync_prepare_coh_trace(x, fs, coh_cfg)
    y, _ = _subthsync_prepare_coh_trace(y, fs, coh_cfg)
    n = min(x.size, y.size)
    x = x[:n]
    y = y[:n]
    if n < 32:
        return None

    nperseg_cfg = int(coh_cfg.get('nperseg', 512))
    nperseg = min(nperseg_cfg, max(32, n // 4))
    nperseg = max(16, min(nperseg, n - 1))

    noverlap_ratio = float(coh_cfg.get('noverlap_ratio', 0.5))
    noverlap = int(np.clip(noverlap_ratio * nperseg, 0, max(0, nperseg - 1)))

    freq, obs_msc = coherence(x, y, fs=float(fs_eff), nperseg=nperseg, noverlap=noverlap)
    obs_msc = np.asarray(obs_msc, dtype=float)

    band = tuple(coh_cfg.get('freq_band_hz', (1.0, 50.0)))
    band_mask = (freq >= float(band[0])) & (freq <= float(band[1]))
    if not np.any(band_mask):
        band_mask = np.isfinite(freq)

    obs_band = float(np.nanmean(obs_msc[band_mask])) if np.any(band_mask) else np.nan
    peak_idx = np.nanargmax(obs_msc[band_mask]) if np.any(band_mask) and np.any(np.isfinite(obs_msc[band_mask])) else None
    peak_hz = np.nan if peak_idx is None else float(freq[band_mask][peak_idx])

    n_shuffle = int(coh_cfg.get('n_shuffle', 80))
    show_progress = bool(coh_cfg.get('show_progress', True))
    progress_leave = bool(coh_cfg.get('progress_leave', False))
    progress_mininterval = float(coh_cfg.get('progress_mininterval_s', 0.2))
    null_specs = []
    null_band = []
    for _ in tqdm(range(max(0, n_shuffle)), desc='MSC shuffles', disable=not show_progress, leave=progress_leave, mininterval=progress_mininterval):
        y_null = _subthsync_coh_null_surrogate(y, fs_eff, coh_cfg, rng)
        _, msc_null = coherence(x, y_null, fs=float(fs_eff), nperseg=nperseg, noverlap=noverlap)
        msc_null = np.asarray(msc_null, dtype=float)
        if msc_null.shape == obs_msc.shape and np.any(np.isfinite(msc_null)):
            null_specs.append(msc_null)
            null_band.append(float(np.nanmean(msc_null[band_mask])) if np.any(band_mask) else np.nan)

    if len(null_specs) == 0:
        p95_curve = np.full_like(obs_msc, np.nan)
        p_band = np.nan
        band_null95 = np.nan
    else:
        null_specs = np.asarray(null_specs, dtype=float)
        p95_curve = np.nanpercentile(null_specs, 95.0, axis=0)
        null_band = np.asarray(null_band, dtype=float)
        null_band = null_band[np.isfinite(null_band)]
        if null_band.size == 0 or not np.isfinite(obs_band):
            p_band = np.nan
            band_null95 = np.nan
        else:
            p_band = float((1.0 + np.sum(null_band >= obs_band)) / (null_band.size + 1.0))
            band_null95 = float(np.nanpercentile(null_band, 95.0))

    band_excess = np.nan if (not np.isfinite(obs_band) or not np.isfinite(band_null95)) else float(obs_band - band_null95)

    return {
        'freq_hz': freq,
        'obs_msc': obs_msc,
        'p95_curve': p95_curve,
        'band_mean': obs_band,
        'band_null95': band_null95,
        'band_excess': band_excess,
        'band_p': p_band,
        'peak_hz': peak_hz,
    }


def _subthsync_bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    finite = np.isfinite(p)
    if not np.any(finite):
        return q
    pf = p[finite]
    order = np.argsort(pf)
    ranked = pf[order]
    m = len(ranked)
    q_ranked = ranked * m / np.arange(1, m + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0.0, 1.0)
    back = np.empty_like(q_ranked)
    back[order] = q_ranked
    q[finite] = back
    return q


def _subthsync_combine_pair_pvalues_fisher(pvals):
    from scipy.stats import chi2

    vals = np.asarray(pvals, dtype=float)
    vals = vals[np.isfinite(vals)]
    vals = vals[(vals > 0.0) & (vals <= 1.0)]
    if vals.size == 0:
        return np.nan
    if vals.size == 1:
        return float(vals[0])
    stat = -2.0 * np.sum(np.log(vals))
    return float(chi2.sf(stat, 2 * vals.size))


def _subthsync_pair_matrix(cell_names_local, pair_df_local, value_col):
    n_local = len(cell_names_local)
    mat = np.full((n_local, n_local), np.nan, dtype=float)
    np.fill_diagonal(mat, np.nan)
    for row in pair_df_local.itertuples(index=False):
        ci = str(row.cell_i)
        cj = str(row.cell_j)
        if ci not in cell_names_local or cj not in cell_names_local:
            continue
        i = cell_names_local.index(ci)
        j = cell_names_local.index(cj)
        val = pd.to_numeric(getattr(row, value_col), errors='coerce')
        if np.isfinite(val):
            mat[i, j] = float(val)
            mat[j, i] = float(val)
    np.fill_diagonal(mat, np.nan)
    return mat


def _subthsync_text_color(val, threshold=None):
    try:
        v = float(val)
    except Exception:
        return 'black'
    if not np.isfinite(v):
        return '#6B7280'
    if threshold is None:
        cfg = globals().get('CORR_HEATMAP_SCALE', {})
        if bool(cfg.get('enabled', False)):
            lim = max(abs(float(cfg.get('vmin', -1.0))), abs(float(cfg.get('vmax', 1.0))))
            threshold = 0.55 * lim
        else:
            threshold = 0.55
    return 'white' if abs(v) >= float(threshold) else 'black'


def _subthsync_format_sig_value(val):
    val = pd.to_numeric(val, errors='coerce')
    if not np.isfinite(val):
        return '=na'
    if val < 0.00001:
        return '<0.00001'
    return f'={float(val):.4f}'


def _subthsync_p_stars(p_val):
    if not np.isfinite(p_val):
        return ''
    if p_val < 0.001:
        return '***'
    if p_val < 0.01:
        return '**'
    if p_val < 0.05:
        return '*'
    return ''


def _subthsync_overlay_text(ax, value_mat, p_mat=None, q_mat=None):
    value_fs = _subthsync_matrix_font('value_fontsize', 16.9)
    stat_fs = _subthsync_matrix_font('stat_fontsize', 11.8)
    for i in range(value_mat.shape[0]):
        for j in range(value_mat.shape[1]):
            val = value_mat[i, j]
            if not np.isfinite(val):
                continue
            x = j + 0.5
            y = i + 0.5
            txt_color = _subthsync_text_color(val)
            if i == j:
                ax.text(x, y, f'{val:.2f}', ha='center', va='center', fontsize=value_fs, fontweight='bold', color=txt_color)
                continue
            p_val = np.nan if p_mat is None else p_mat[i, j]
            q_val = np.nan if q_mat is None else q_mat[i, j]
            stars = _subthsync_p_stars(p_val)
            ax.text(x, y - 0.12, f'{val:.2f}{stars}', ha='center', va='center', fontsize=value_fs, fontweight='bold', color=txt_color)
            ax.text(x, y + 0.10, f'p{_subthsync_format_sig_value(p_val)}', ha='center', va='center', fontsize=stat_fs, color=txt_color)
            if np.isfinite(q_val):
                ax.text(x, y + 0.24, f'q{_subthsync_format_sig_value(q_val)}', ha='center', va='center', fontsize=stat_fs, color=txt_color)


def _subthsync_style_matrix_axis(ax, title):
    tick_fs = _subthsync_matrix_font('tick_label_fontsize', 15.2)
    title_fs = _subthsync_matrix_font('title_fontsize', 18.6)
    label_fs = _subthsync_matrix_font('colorbar_label_fontsize', 16.9)
    ax.set_title(title, fontsize=title_fs)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=tick_fs)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=tick_fs)
    if getattr(ax, 'collections', None):
        cbar = ax.collections[0].colorbar
        if cbar is not None:
            cbar_tick_fs = _subthsync_matrix_font('colorbar_tick_fontsize', 8)
            cbar.ax.tick_params(labelsize=cbar_tick_fs)
            cbar.set_label(cbar.ax.get_ylabel(), size=label_fs)


if not bool(SUBTH_SYNC_CONFIG.get('enabled', True)):
    print('Subthreshold synchrony section is disabled. Set SUBTH_SYNC_CONFIG["enabled"] = True to show it.')
elif n_cells < 2:
    print('Single-cell dataset detected (n_cells=1). Skipping subthreshold synchrony matrices.')
else:
    rng = np.random.default_rng(int(SUBTH_SYNC_CONFIG.get('rng_seed', 123)))
    coh_rng = np.random.default_rng(int(COHERENCE_CONFIG.get('rng_seed', 123)))
    subth_sync_rows = []
    mask_post_event_period = bool(SUBTH_SYNC_CONFIG.get('mask_post_event_period', False))
    sync_suffix = ' (post-event response masked)' if mask_post_event_period else ''

    if mask_post_event_period:
        print(f'Subthreshold whole-recording synchrony masks EVENT_RESPONSE_MS={EVENT_RESPONSE_MS} around each event before pooling.')

    pooled_pair_traces = {}
    for i in range(n_cells):
        for j in range(i + 1, n_cells):
            pooled_pair_traces[(i, j)] = {'a': [], 'b': [], 'fs': []}

    for s in processed_sessions:
        raw_subth = np.asarray(s['subthreshold'], dtype=float)
        fs = float(s['fs'])
        if mask_post_event_period:
            event_times_ms = _subthsync_event_times_for_session(s)
            subth = np.column_stack([
                _subthsync_mask_trace(raw_subth[:, cell_idx], s['time_ms'], event_times_ms, EVENT_RESPONSE_MS)
                for cell_idx in range(s['n_cells'])
            ])
        else:
            subth = raw_subth

        for i in range(s['n_cells']):
            for j in range(i + 1, s['n_cells']):
                obs_r, p_val, z_val = _subthsync_shuffle_significance(
                    subth[:, i], subth[:, j], fs, SUBTH_SYNC_CONFIG, rng
                )

                subth_sync_rows.append({
                    'session': s['session_name'],
                    'cell_i': s['cell_names'][i],
                    'cell_j': s['cell_names'][j],
                    'i': i,
                    'j': j,
                    'r': obs_r,
                    'p': p_val,
                    'z': z_val,
                })

                pooled_pair_traces[(i, j)]['a'].append(np.asarray(subth[:, i], dtype=float))
                pooled_pair_traces[(i, j)]['b'].append(np.asarray(subth[:, j], dtype=float))
                pooled_pair_traces[(i, j)]['fs'].append(fs)

    pair_coh_stats = {}
    if bool(COHERENCE_CONFIG.get('enabled', True)):
        for (i, j), chunks in pooled_pair_traces.items():
            xa = np.concatenate(chunks['a']) if len(chunks['a']) else np.array([], dtype=float)
            xb = np.concatenate(chunks['b']) if len(chunks['b']) else np.array([], dtype=float)
            fs_ref = float(np.nanmedian(np.asarray(chunks['fs'], dtype=float))) if len(chunks['fs']) else np.nan
            pair_coh_stats[(i, j)] = _subthsync_coherence_with_shuffle(xa, xb, fs_ref, COHERENCE_CONFIG, coh_rng)

    subth_sync_df = pd.DataFrame(subth_sync_rows)

    pooled_rows = []
    for (cell_i, cell_j), grp in subth_sync_df.groupby(['cell_i', 'cell_j'], sort=False):
        vals = pd.to_numeric(grp['r'], errors='coerce').dropna().to_numpy(dtype=float)
        p_vals = pd.to_numeric(grp['p'], errors='coerce').dropna().to_numpy(dtype=float)

        i = int(grp['i'].iloc[0])
        j = int(grp['j'].iloc[0])
        coh_stats = pair_coh_stats.get((min(i, j), max(i, j)), None)
        pooled_chunks = pooled_pair_traces.get((min(i, j), max(i, j)), {'a': [], 'b': [], 'fs': []})
        pooled_a = np.concatenate(pooled_chunks['a']) if len(pooled_chunks['a']) else np.array([], dtype=float)
        pooled_b = np.concatenate(pooled_chunks['b']) if len(pooled_chunks['b']) else np.array([], dtype=float)
        pooled_fs = float(np.nanmedian(np.asarray(pooled_chunks['fs'], dtype=float))) if len(pooled_chunks['fs']) else np.nan
        pooled_r, pooled_p, pooled_z = _subthsync_shuffle_significance(
            pooled_a,
            pooled_b,
            pooled_fs,
            SUBTH_SYNC_CONFIG,
            rng,
        )

        pooled_rows.append({
            'cell_i': cell_i,
            'cell_j': cell_j,
            'mean_metric': pooled_r,
            'session_mean_metric': float(np.nanmean(vals)) if vals.size else np.nan,
            'session_median_metric': float(np.nanmedian(vals)) if vals.size else np.nan,
            'p': pooled_p,
            'z': pooled_z,
            'p_fisher_legacy': _subthsync_combine_pair_pvalues_fisher(p_vals),
            'msc_band_mean': np.nan if coh_stats is None else coh_stats['band_mean'],
            'msc_band_null95': np.nan if coh_stats is None else coh_stats['band_null95'],
            'msc_band_excess': np.nan if coh_stats is None else coh_stats['band_excess'],
            'p_msc': np.nan if coh_stats is None else coh_stats['band_p'],
            'n_sessions': int(grp['session'].nunique()),
        })

    pooled_subth_sync_summary = pd.DataFrame(pooled_rows)
    if not pooled_subth_sync_summary.empty:
        pooled_subth_sync_summary['q'] = _subthsync_bh_fdr(pooled_subth_sync_summary['p'].values)
        pooled_subth_sync_summary['q_msc'] = _subthsync_bh_fdr(pooled_subth_sync_summary['p_msc'].values)

    clear_fn = globals().get('clear_output', None)
    if bool(NOTEBOOK_OUTPUT_CONFIG.get('clear_intermediate_progress', True)) and callable(clear_fn):
        clear_fn(wait=True)
    if mask_post_event_period:
        print(f'Subthreshold synchrony masked EVENT_RESPONSE_MS={EVENT_RESPONSE_MS} around each event before pooling.')
    display_top(subth_sync_df, label='Session-level pair synchrony', sort_by='r')

    print('Pooled subthreshold synchrony summary (Pearson + MSC):')
    display_top(pooled_subth_sync_summary, label='Pooled subthreshold synchrony summary', sort_by='mean_metric')

    pooled_subth_sync_mat = _subthsync_pair_matrix(cell_names, pooled_subth_sync_summary, 'mean_metric')
    pooled_subth_p_mat = _subthsync_pair_matrix(cell_names, pooled_subth_sync_summary, 'p')
    pooled_subth_q_mat = _subthsync_pair_matrix(cell_names, pooled_subth_sync_summary, 'q')

    pooled_subth_msc_mat = _subthsync_pair_matrix(cell_names, pooled_subth_sync_summary, 'msc_band_excess')
    pooled_subth_msc_p = _subthsync_pair_matrix(cell_names, pooled_subth_sync_summary, 'p_msc')
    pooled_subth_msc_q = _subthsync_pair_matrix(cell_names, pooled_subth_sync_summary, 'q_msc')

    # Keep large-ROI matrices readable without producing an oversized PNG.
    fig_w = max(7.5, min(14.0, 0.55 * n_cells + 4.0))
    fig_h = max(5.8, min(12.0, 0.55 * n_cells + 2.0))

    fig, axes = plt.subplots(1, 2, figsize=(2.05 * fig_w, fig_h), constrained_layout=True)

    sns.heatmap(
        pooled_subth_sync_mat,
        ax=axes[0],
        annot=False,
        fmt='',
        cmap='vlag',
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        xticklabels=cell_names,
        yticklabels=cell_names,
        linewidths=0.5,
        linecolor='#E5E7EB',
        cbar_kws={'shrink': 0.85, 'label': 'Pearson r'},
    )
    axes[0].set_aspect('equal', adjustable='box')
    _subthsync_overlay_text(axes[0], pooled_subth_sync_mat, pooled_subth_p_mat, pooled_subth_q_mat)
    _subthsync_style_matrix_axis(
        axes[0],
        f'Pooled subthreshold synchrony (pooled Pearson r){sync_suffix}\n'
    )

    sns.heatmap(
        pooled_subth_msc_mat,
        ax=axes[1],
        annot=False,
        fmt='',
        cmap='vlag',
        center=0,
        vmin=-0.35,
        vmax=0.35,
        square=True,
        xticklabels=cell_names,
        yticklabels=cell_names,
        linewidths=0.5,
        linecolor='#E5E7EB',
        cbar_kws={'shrink': 0.85, 'label': 'MSC excess (obs - null95)'},
    )
    axes[1].set_aspect('equal', adjustable='box')
    _subthsync_overlay_text(axes[1], pooled_subth_msc_mat, pooled_subth_msc_p, pooled_subth_msc_q)
    _subthsync_style_matrix_axis(
        axes[1],
        f'Pooled subthreshold synchrony (MSC excess vs null95){sync_suffix}\n'
    )

    for ax in axes:
        sns.despine(ax=ax, left=True, bottom=True)
    plt.show()

    if bool(COHERENCE_CONFIG.get('enabled', True)) and bool(COHERENCE_CONFIG.get('show_pair_spectra', True)):
        print('Pair coherence spectra with shuffle-derived 95th percentile thresholds:')
        max_f = float(COHERENCE_CONFIG.get('max_plot_freq_hz', 120.0))
        y_max = float(COHERENCE_CONFIG.get('plot_y_max', 0.5))
        use_log_x = bool(COHERENCE_CONFIG.get('plot_log_x', False))
        all_pairs = [(i, j) for i in range(n_cells) for j in range(i + 1, n_cells)]
        ranked_pairs = sorted(
            all_pairs,
            key=lambda p: abs(float(pair_coh_stats.get(p, {}).get('band_excess', np.nan))) if pair_coh_stats.get(p, None) is not None and np.isfinite(pair_coh_stats[p].get('band_excess', np.nan)) else -1.0,
            reverse=True,
        )
        max_plot_pairs = min(int(COHERENCE_CONFIG.get('max_plot_pairs', len(ranked_pairs))), int(NOTEBOOK_OUTPUT_CONFIG['pair_plot_limit']))
        for i, j in ranked_pairs[:max_plot_pairs]:
            coh_stats = pair_coh_stats.get((i, j), None)
            if coh_stats is None:
                continue

            f = np.asarray(coh_stats['freq_hz'], dtype=float)
            m = np.asarray(coh_stats['obs_msc'], dtype=float)
            p95 = np.asarray(coh_stats['p95_curve'], dtype=float)
            mask_f = np.isfinite(f) & (f <= max_f)
            if not np.any(mask_f):
                continue

            fig, ax = plt.subplots(1, 1, figsize=(8.8, 3.8), constrained_layout=True)
            ax.plot(f[mask_f], m[mask_f], color='#1F77B4', lw=2.0, label='Observed MSC')
            if np.any(np.isfinite(p95[mask_f])):
                ax.plot(f[mask_f], p95[mask_f], color='#D62728', ls='--', lw=1.8, label='Shuffle 95th percentile')
            ax.set_ylim(0, max(0.05, y_max))
            if use_log_x:
                pos_f = f[mask_f][f[mask_f] > 0]
                if pos_f.size:
                    ax.set_xscale('log')
                    ax.set_xlim(max(0.1, float(np.nanmin(pos_f))), max_f)
                else:
                    ax.set_xlim(0, max_f)
            else:
                ax.set_xlim(0, max_f)
            ax.set_title(f"{cell_names[i]} <-> {cell_names[j]} | Coherence spectrum")
            ax.set_xlabel('Frequency (Hz)')
            ax.set_ylabel('Magnitude-squared coherence')
            ax.legend(frameon=False, fontsize=9)
            sns.despine(ax=ax, left=True, bottom=True)
            plt.show()

    if bool(SUBTH_SYNC_CONFIG.get('show_per_session_matrices', False)):
        for s in processed_sessions:
            sub = subth_sync_df[subth_sync_df['session'] == s['session_name']].copy()
            if sub.empty:
                continue
            sub['q'] = _subthsync_bh_fdr(pd.to_numeric(sub['p'], errors='coerce').values)

            mat_r = _subthsync_pair_matrix(cell_names, sub, 'r')
            p_mat_r = _subthsync_pair_matrix(cell_names, sub, 'p')
            q_mat_r = _subthsync_pair_matrix(cell_names, sub, 'q')

            mat_msc = np.full_like(mat_r, np.nan, dtype=float)
            for i in range(s['n_cells']):
                for j in range(i + 1, s['n_cells']):
                    raw_subth = np.asarray(s['subthreshold'], dtype=float)
                    if mask_post_event_period:
                        event_times_ms = _subthsync_event_times_for_session(s)
                        xa = _subthsync_mask_trace(raw_subth[:, i], s['time_ms'], event_times_ms, EVENT_RESPONSE_MS)
                        xb = _subthsync_mask_trace(raw_subth[:, j], s['time_ms'], event_times_ms, EVENT_RESPONSE_MS)
                    else:
                        xa = raw_subth[:, i]
                        xb = raw_subth[:, j]
                    quick_stats = _subthsync_coherence_with_shuffle(
                        xa,
                        xb,
                        float(s['fs']),
                        {**COHERENCE_CONFIG, 'n_shuffle': 0},
                        coh_rng,
                    )
                    if quick_stats is not None and np.isfinite(quick_stats['band_mean']):
                        mat_msc[i, j] = quick_stats['band_mean']
                        mat_msc[j, i] = quick_stats['band_mean']

            fig, axes = plt.subplots(1, 2, figsize=(2.05 * fig_w, fig_h), constrained_layout=True)

            sns.heatmap(
                mat_r,
                ax=axes[0],
                annot=False,
                fmt='',
                cmap='vlag',
                center=0,
                vmin=-1,
                vmax=1,
                square=True,
                xticklabels=cell_names,
                yticklabels=cell_names,
                linewidths=0.5,
                linecolor='#E5E7EB',
                cbar_kws={'shrink': 0.85, 'label': 'Pearson r'},
            )
            axes[0].set_aspect('equal', adjustable='box')
            _subthsync_overlay_text(axes[0], mat_r, p_mat_r, q_mat_r)
            _subthsync_style_matrix_axis(
                axes[0],
                f'{s["session_name"]}: subthreshold synchrony (Pearson){sync_suffix}\n'
            )

            sns.heatmap(
                mat_msc,
                ax=axes[1],
                annot=False,
                fmt='',
                cmap='vlag',
                center=0,
                vmin=0,
                vmax=1,
                square=True,
                xticklabels=cell_names,
                yticklabels=cell_names,
                linewidths=0.5,
                linecolor='#E5E7EB',
                cbar_kws={'shrink': 0.85, 'label': 'MSC band mean'},
            )
            axes[1].set_aspect('equal', adjustable='box')
            _subthsync_overlay_text(axes[1], mat_msc, None, None)
            _subthsync_style_matrix_axis(
                axes[1],
                f'{s["session_name"]}: subthreshold synchrony (MSC quick){sync_suffix}\n'
            )

            for ax in axes:
                sns.despine(ax=ax, left=True, bottom=True)
            plt.show()

In [ ]:
# ==============================
# SS IFR coherence / MSC (whole recording)
# ==============================
def _ss_ifr_from_spikes(spike_times_ms, fs, n_samples, sigma_ms, scale_to_hz=True):
    x = np.zeros(int(n_samples), dtype=float)
    st = np.asarray(spike_times_ms, dtype=float)
    if st.size > 0:
        idx = np.round(st * float(fs) / 1000.0).astype(int)
        idx = idx[(idx >= 0) & (idx < int(n_samples))]
        if idx.size:
            np.add.at(x, idx, 1.0)

    sigma_samples = max(0.0, float(sigma_ms) * float(fs) / 1000.0)
    if sigma_samples > 0:
        x = gaussian_filter1d(x, sigma=sigma_samples, mode='nearest')

    if bool(scale_to_hz):
        x = x * float(fs)
    return x


def _gaussian_gain_at_freq(freq_hz, sigma_ms):
    sigma_s = max(1e-12, float(sigma_ms) / 1000.0)
    w = 2.0 * np.pi * float(freq_hz)
    return float(np.exp(-0.5 * (w * sigma_s) ** 2))


def _gaussian_f3db_hz(sigma_ms):
    sigma_s = max(1e-12, float(sigma_ms) / 1000.0)
    return float(np.sqrt(np.log(2.0)) / (2.0 * np.pi * sigma_s))


if not bool(SS_MSC_CONFIG.get('enabled', True)):
    print("SS IFR coherence section is disabled. Set SS_MSC_CONFIG['enabled'] = True to run.")
elif n_cells < 2:
    print('Single-cell dataset detected (n_cells=1). Skipping SS IFR coherence.')
else:
    if '_subthsync_coherence_with_shuffle' not in globals():
        raise RuntimeError('Run Cell 10 first so coherence helper functions are defined.')

    sigma_ms = float(SS_MSC_CONFIG.get('ifreq_sigma_ms', 5.0))
    print('SS IFR smoothing notes:')
    print(f"- Gaussian sigma: {sigma_ms:.2f} ms")
    print(f"- Approx Gaussian -3 dB frequency: {_gaussian_f3db_hz(sigma_ms):.2f} Hz")
    print(f"- Gain at 50 Hz: {_gaussian_gain_at_freq(50.0, sigma_ms):.3f}")
    print(f"- Gain at 100 Hz: {_gaussian_gain_at_freq(100.0, sigma_ms):.3f}")

    ss_cfg = {
        'enabled': True,
        'freq_band_hz': tuple(SS_MSC_CONFIG.get('freq_band_hz', (0.2, 100.0))),
        'nperseg': int(SS_MSC_CONFIG.get('nperseg', 1024)),
        'noverlap_ratio': float(SS_MSC_CONFIG.get('noverlap_ratio', 0.5)),
        'n_shuffle': int(SS_MSC_CONFIG.get('n_shuffle', 100)),
        'shuffle_mode': str(SS_MSC_CONFIG.get('shuffle_mode', 'circular_shift')),
        'jitter_ms': float(SS_MSC_CONFIG.get('jitter_ms', 200.0)),
        'min_shift_ms': float(SS_MSC_CONFIG.get('min_shift_ms', 100.0)),
        'rng_seed': int(SS_MSC_CONFIG.get('rng_seed', 123)),
        'max_plot_freq_hz': float(SS_MSC_CONFIG.get('max_plot_freq_hz', 100.0)),
        'max_signal_samples': int(SS_MSC_CONFIG.get('max_signal_samples', 150000)),
        'max_fs_hz': float(SS_MSC_CONFIG.get('max_fs_hz', 300.0)),
    }

    ss_rng = np.random.default_rng(int(SS_MSC_CONFIG.get('rng_seed', 123)))
    mask_post_event_period = bool(SS_MSC_CONFIG.get('mask_post_event_period', True))
    ss_pair_stats = {}

    pooled_ss_ifr = {(i, j): {'a': [], 'b': [], 'fs': []} for i in range(n_cells) for j in range(i + 1, n_cells)}

    for s in processed_sessions:
        fs = float(s['fs'])
        n_samples = int(len(s['time_ms']))

        ifr_mat = np.column_stack([
            _ss_ifr_from_spikes(
                s['spike_times_ss'][cell_idx],
                fs,
                n_samples,
                sigma_ms=sigma_ms,
                scale_to_hz=bool(SS_MSC_CONFIG.get('scale_to_hz', True)),
            )
            for cell_idx in range(s['n_cells'])
        ])

        if mask_post_event_period:
            event_times_ms = _subthsync_event_times_for_session(s)
            ifr_mat = np.column_stack([
                _subthsync_mask_trace(ifr_mat[:, cell_idx], s['time_ms'], event_times_ms, EVENT_RESPONSE_MS)
                for cell_idx in range(s['n_cells'])
            ])

        for i in range(s['n_cells']):
            for j in range(i + 1, s['n_cells']):
                pooled_ss_ifr[(i, j)]['a'].append(np.asarray(ifr_mat[:, i], dtype=float))
                pooled_ss_ifr[(i, j)]['b'].append(np.asarray(ifr_mat[:, j], dtype=float))
                pooled_ss_ifr[(i, j)]['fs'].append(fs)

    rows = []
    for (i, j), chunks in pooled_ss_ifr.items():
        xa = np.concatenate(chunks['a']) if len(chunks['a']) else np.array([], dtype=float)
        xb = np.concatenate(chunks['b']) if len(chunks['b']) else np.array([], dtype=float)
        fs_ref = float(np.nanmedian(np.asarray(chunks['fs'], dtype=float))) if len(chunks['fs']) else np.nan

        stats = _subthsync_coherence_with_shuffle(xa, xb, fs_ref, ss_cfg, ss_rng)
        ss_pair_stats[(i, j)] = stats
        rows.append({
            'cell_i': cell_names[i],
            'cell_j': cell_names[j],
            'n_sessions': int(len(chunks['fs'])),
            'msc_band_mean': np.nan if stats is None else stats['band_mean'],
            'msc_band_null95': np.nan if stats is None else stats['band_null95'],
            'msc_band_excess': np.nan if stats is None else stats['band_excess'],
            'p_msc': np.nan if stats is None else stats['band_p'],
            'peak_hz': np.nan if stats is None else stats['peak_hz'],
        })

    ss_msc_df = pd.DataFrame(rows)
    if not ss_msc_df.empty:
        ss_msc_df['q_msc'] = _subthsync_bh_fdr(ss_msc_df['p_msc'].values)

    print('SS IFR coherence summary (pooled):')
    display_top(ss_msc_df, label='SS IFR coherence summary', sort_by='msc_band_excess')

    if not ss_msc_df.empty:
        mat_excess = _subthsync_pair_matrix(cell_names, ss_msc_df, 'msc_band_excess')
        p_mat = _subthsync_pair_matrix(cell_names, ss_msc_df, 'p_msc')
        q_mat = _subthsync_pair_matrix(cell_names, ss_msc_df, 'q_msc')

        fig, ax = plt.subplots(1, 1, figsize=(8.3, 6.2), constrained_layout=True)
        sns.heatmap(
            mat_excess,
            ax=ax,
            annot=False,
            fmt='',
            cmap='vlag',
            center=0,
            vmin=-0.35,
            vmax=0.35,
            square=True,
            xticklabels=cell_names,
            yticklabels=cell_names,
            linewidths=0.5,
            linecolor='#E5E7EB',
            cbar_kws={'shrink': 0.85, 'label': 'SS IFR MSC excess (obs - null95)'},
        )
        ax.set_aspect('equal', adjustable='box')
        _subthsync_overlay_text(ax, mat_excess, p_mat, q_mat)
        _subthsync_style_matrix_axis(ax, 'SS spike-train coherence (IFR) MSC excess')
        sns.despine(ax=ax, left=True, bottom=True)
        plt.show()

    if bool(SS_MSC_CONFIG.get('show_pair_spectra', True)):
        print('SS IFR pair coherence spectra:')
        max_f = float(SS_MSC_CONFIG.get('max_plot_freq_hz', 100.0))
        y_max = float(SS_MSC_CONFIG.get('plot_y_max', 0.5))
        use_log_x = bool(SS_MSC_CONFIG.get('plot_log_x', False))
        max_plot_pairs = min(int(SS_MSC_CONFIG.get('max_plot_pairs', len(ss_pair_stats))), int(NOTEBOOK_OUTPUT_CONFIG['pair_plot_limit']))

        ranked_pairs = sorted(
            [(i, j) for i in range(n_cells) for j in range(i + 1, n_cells)],
            key=lambda p: abs(float(ss_pair_stats.get(p, {}).get('band_excess', np.nan))) if ss_pair_stats.get(p, None) is not None and np.isfinite(ss_pair_stats[p].get('band_excess', np.nan)) else -1.0,
            reverse=True,
        )

        for i, j in ranked_pairs[:max_plot_pairs]:
            stats = ss_pair_stats.get((i, j), None)
            if stats is None:
                continue

            f = np.asarray(stats['freq_hz'], dtype=float)
            m = np.asarray(stats['obs_msc'], dtype=float)
            p95 = np.asarray(stats['p95_curve'], dtype=float)
            mask_f = np.isfinite(f) & (f <= max_f)
            if not np.any(mask_f):
                continue

            fig, ax = plt.subplots(1, 1, figsize=(8.8, 3.8), constrained_layout=True)
            ax.plot(f[mask_f], m[mask_f], color='#1F77B4', lw=2.0, label='Observed MSC')
            if np.any(np.isfinite(p95[mask_f])):
                ax.plot(f[mask_f], p95[mask_f], color='#D62728', ls='--', lw=1.8, label='Shuffle 95th percentile')

            ax.set_ylim(0, max(0.05, y_max))
            if use_log_x:
                pos_f = f[mask_f][f[mask_f] > 0]
                if pos_f.size:
                    ax.set_xscale('log')
                    ax.set_xlim(max(0.1, float(np.nanmin(pos_f))), max_f)
                else:
                    ax.set_xlim(0, max_f)
            else:
                ax.set_xlim(0, max_f)

            ax.set_title(f"{cell_names[i]} <-> {cell_names[j]} | SS IFR coherence")
            ax.set_xlabel('Frequency (Hz)')
            ax.set_ylabel('Magnitude-squared coherence')
            ax.legend(frameon=False, fontsize=9)
            sns.despine(ax=ax, left=True, bottom=True)
            plt.show()

In [ ]:
# ==============================

# Subthreshold <-> SS/CS relation

# ==============================

def _valid_spike_indices(spike_times_ms, fs, trace_len):

    idx = np.round(np.asarray(spike_times_ms, dtype=float) * fs / 1000.0).astype(int)

    return idx[(idx >= 0) & (idx < int(trace_len))]



def _extract_spike_snippets(trace, spike_times_ms, fs, window_ms=(-150, 150), allow_partial=True):

    pre_s = int(round((window_ms[0] / 1000.0) * fs))

    post_s = int(round((window_ms[1] / 1000.0) * fs))

    if post_s <= pre_s:

        raise ValueError('Invalid STA window')

    trace = np.asarray(trace, dtype=float)

    idx = _valid_spike_indices(spike_times_ms, fs, len(trace))

    if len(idx) == 0:

        return None, None

    snippet_len = post_s - pre_s + 1

    snippets = []

    for p in idx:

        a = p + pre_s

        b = p + post_s + 1

        if (a < 0 or b > len(trace)) and (not allow_partial):

            continue

        snippet = np.full(snippet_len, np.nan, dtype=float)

        src_a = max(0, a)

        src_b = min(len(trace), b)

        if src_b <= src_a:

            continue

        dst_a = src_a - a

        dst_b = dst_a + (src_b - src_a)

        snippet[dst_a:dst_b] = trace[src_a:src_b]

        snippets.append(snippet)

    if len(snippets) == 0:

        return None, None

    arr = np.vstack(snippets)

    rel_t = np.arange(pre_s, post_s + 1) * 1000.0 / fs

    return rel_t, arr



def _nansem(arr, axis=0):

    arr = np.asarray(arr, dtype=float)

    if arr.ndim != 2:

        return np.zeros(0, dtype=float)

    valid = np.sum(np.isfinite(arr), axis=axis)

    with np.errstate(invalid='ignore', divide='ignore'):

        sem = np.nanstd(arr, axis=axis, ddof=1) / np.sqrt(valid)

    sem = np.asarray(sem, dtype=float)

    sem[~np.isfinite(sem)] = 0.0

    sem[valid < 2] = 0.0

    return sem



def spike_triggered_average(trace, time_ms, spike_times_ms, fs, window_ms=(-150, 150)):

    rel_t, arr = _extract_spike_snippets(trace, spike_times_ms, fs, window_ms=window_ms, allow_partial=True)

    if arr is None:

        return None, None, 0

    sta = np.nanmean(arr, axis=0)

    sem = _nansem(arr, axis=0)

    return rel_t, (sta, sem), arr.shape[0]



def spike_triggered_snippets(trace, spike_times_ms, fs, window_ms=(-150, 150)):

    return _extract_spike_snippets(trace, spike_times_ms, fs, window_ms=window_ms, allow_partial=True)



def _sta_mask_window_for_type(stype):

    if str(stype).upper() == 'SS':

        return tuple(PREPROCESS_CONFIG.get('spike_mask_ss_ms', SPIKE_MASK_SS_MS))

    return tuple(PREPROCESS_CONFIG.get('spike_mask_cs_ms', SPIKE_MASK_CS_MS))


def _sta_add_mask_overlay(ax, stype):

    if not bool(globals().get('STA_SHOW_MASK_OVERLAY', True)):

        return

    w0, w1 = _sta_mask_window_for_type(stype)

    lo, hi = sorted((float(w0), float(w1)))

    if not (np.isfinite(lo) and np.isfinite(hi)) or lo == hi:

        return

    ax.axvspan(lo, hi, color=str(globals().get('STA_MASK_OVERLAY_COLOR', '#9CA3AF')), alpha=float(globals().get('STA_MASK_OVERLAY_ALPHA', 0.2)), zorder=0, lw=0)


rows = []

for s in processed_sessions:

    t = s['time_ms']

    fs = float(s['fs'])

    for c in range(s['n_cells']):

        tr = s['subthreshold'][:, c]

        for stype, spikes, sta_w in [
            ('SS', s['spike_times_ss'][c], STA_WINDOW_SS_MS),
            ('CS', s['spike_times_cs'][c], STA_WINDOW_CS_MS),
        ]:

            sp = np.asarray(spikes, dtype=float)

            if len(sp) < MIN_SPIKES_FOR_STA:

                continue

            idx = _valid_spike_indices(sp, fs, len(tr))

            if len(idx) == 0:

                continue

            amp_at_spike = tr[idx]

            slope_idx = idx[idx >= 1]

            pre_slope = tr[slope_idx] - tr[slope_idx - 1] if len(slope_idx) else np.array([np.nan])

            rows.append({

                'session': s['session_name'],

                'cell': s['cell_names'][c],

                'spike_type': stype,

                'n_spikes': int(len(idx)),

                'subth_at_spike_mean': float(np.nanmean(amp_at_spike)),

                'subth_at_spike_std': float(np.nanstd(amp_at_spike, ddof=1)) if len(amp_at_spike) > 1 else 0.0,

                'pre_spike_slope_mean': float(np.nanmean(pre_slope)),

            })

spike_subth_df = pd.DataFrame(rows)

display(spike_subth_df.head(20))



# Example STA on selected session/cell

s = processed_sessions[EXAMPLE_SESSION_IDX]

c = EXAMPLE_CELL_IDX

tr = s['subthreshold'][:, c]

fs = float(s['fs'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, constrained_layout=True)

for ax, stype, spikes, sta_w in [

    (axes[0], 'SS', s['spike_times_ss'][c], STA_WINDOW_SS_MS),

    (axes[1], 'CS', s['spike_times_cs'][c], STA_WINDOW_CS_MS),

]:

    rel_t, out, n = spike_triggered_average(tr, s['time_ms'], spikes, fs, window_ms=sta_w)

    if out is None:

        ax.set_title(f'{stype} STA (insufficient spikes)')

        sns.despine(ax=ax, left=True, bottom=True)

        continue

    sta, sem = out

    color = SUBTH_COLORS['ss'] if stype == 'SS' else SUBTH_COLORS['cs']

    ax.plot(rel_t, sta, lw=2, color=color)

    ax.fill_between(rel_t, sta - sem, sta + sem, alpha=0.2, color=color)

    _sta_add_mask_overlay(ax, stype)

    ax.axvline(0, ls='--', color='k', lw=1)

    ax.set_title(f'{stype} STA (n={n}, window={sta_w[0]:.0f} to {sta_w[1]:.0f} ms)')

    ax.set_xlabel('Time from spike (ms)')

    sns.despine(ax=ax, left=True, bottom=True)

axes[0].set_ylabel('Subthreshold raw value')

plt.show()



# Pooled STA for every cell across sessions (SS and CS)

sta_snippets_by_cell = {i: {'SS': [], 'CS': []} for i in range(n_cells)}

pooled_sta_blocks = {'SS': [], 'CS': []}

rel_t_sta_ref = {'SS': None, 'CS': None}

for s in processed_sessions:

    fs = float(s['fs'])

    for c in range(s['n_cells']):

        tr = s['subthreshold'][:, c]

        for stype, spikes, sta_w in [
            ('SS', s['spike_times_ss'][c], STA_WINDOW_SS_MS),
            ('CS', s['spike_times_cs'][c], STA_WINDOW_CS_MS),
        ]:

            rel_t_sta, arr = spike_triggered_snippets(tr, spikes, fs, window_ms=sta_w)

            if arr is None:

                continue

            if rel_t_sta_ref[stype] is None:

                rel_t_sta_ref[stype] = rel_t_sta.copy()

            elif len(rel_t_sta) != len(rel_t_sta_ref[stype]) or (not np.allclose(rel_t_sta, rel_t_sta_ref[stype], atol=1e-9)):

                continue

            sta_snippets_by_cell[c][stype].append(arr)

            pooled_sta_blocks[stype].append(arr)



if rel_t_sta_ref['SS'] is None and rel_t_sta_ref['CS'] is None:

    print('No valid STA snippets available for pooled per-cell SS/CS plots.')

else:

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=False, sharey=False, constrained_layout=True)

    support_rows = []

    for ax, stype in zip(axes, ['SS', 'CS']):

        blocks = pooled_sta_blocks[stype]

        color = SUBTH_COLORS['ss'] if stype == 'SS' else SUBTH_COLORS['cs']

        if len(blocks) == 0 or rel_t_sta_ref[stype] is None:

            ax.set_title(f'Pooled {stype} STA (no spikes)')

            ax.axvline(0, ls='--', color='k', lw=1)

            sns.despine(ax=ax, left=True, bottom=True)

            support_rows.append({'spike_type': stype, 'pooled_n': 0, 'sum_of_per_cell_n': 0})

            continue

        pooled = np.vstack(blocks)

        sta = np.nanmean(pooled, axis=0)

        sem = _nansem(pooled, axis=0)

        per_cell_total = int(sum(np.vstack(cell_blocks).shape[0] for cell_blocks in [sta_snippets_by_cell[c][stype] for c in range(n_cells)] if len(cell_blocks) > 0))

        support_rows.append({'spike_type': stype, 'pooled_n': int(pooled.shape[0]), 'sum_of_per_cell_n': per_cell_total})

        ax.plot(rel_t_sta_ref[stype], sta, lw=2, color=color)

        ax.fill_between(rel_t_sta_ref[stype], sta - sem, sta + sem, color=color, alpha=0.2)

        _sta_add_mask_overlay(ax, stype)

        ax.axvline(0, ls='--', color='k', lw=1)

        ax.set_title(f'Pooled {stype} STA across all cells (n={pooled.shape[0]})')

        ax.set_xlabel('Time from spike (ms)')

        sns.despine(ax=ax, left=True, bottom=True)

    axes[0].set_ylabel('Subthreshold raw value')

    plt.show()

    support_counts = pd.DataFrame(support_rows)

    display_top(support_counts, label='STA support counts')

    fig, axes = plt.subplots(n_cells, 2, figsize=(12, max(3, 2.8 * n_cells)), sharex=False, sharey=False, constrained_layout=True)

    axes = np.asarray(axes)

    if axes.ndim == 1:

        axes = axes[None, :]

    for c in range(n_cells):

        for col, (stype, sta_w) in enumerate([('SS', STA_WINDOW_SS_MS), ('CS', STA_WINDOW_CS_MS)]):

            ax = axes[c, col]

            blocks = sta_snippets_by_cell[c][stype]

            color = SUBTH_COLORS['ss'] if stype == 'SS' else SUBTH_COLORS['cs']

            if len(blocks) == 0 or rel_t_sta_ref[stype] is None:

                ax.set_title(f"{cell_names[c]} | {stype} STA (no spikes)")

                ax.axvline(0, ls='--', color='k', lw=1)

                sns.despine(ax=ax, left=True, bottom=True)

                continue

            pooled = np.vstack(blocks)

            sta = np.nanmean(pooled, axis=0)

            sem = _nansem(pooled, axis=0)

            ax.plot(rel_t_sta_ref[stype], sta, lw=1.8, color=color)

            ax.fill_between(rel_t_sta_ref[stype], sta - sem, sta + sem, color=color, alpha=0.2)

            _sta_add_mask_overlay(ax, stype)

            ax.axvline(0, ls='--', color='k', lw=1)

            ax.set_title(f"{cell_names[c]} | {stype} STA (n={pooled.shape[0]})")

            sns.despine(ax=ax, left=True, bottom=True)

            if col == 0:

                ax.set_ylabel('Subthreshold raw value')

            if c == n_cells - 1:

                ax.set_xlabel('Time from spike (ms)')

    fig.suptitle('', y=1.02)

    plt.show()


In [ ]:
# ==============================
# Subthreshold <-> event relation
# ==============================
def event_locked_subthreshold(trace, time_ms, event_times_ms, window_ms=(-600, 1200)):
    dt = float(np.median(np.diff(time_ms)))
    pre_s = int(round(window_ms[0] / dt))
    post_s = int(round(window_ms[1] / dt))

    snippets = []
    for ev in np.asarray(event_times_ms, dtype=float):
        p = int(np.argmin(np.abs(time_ms - ev)))
        a, b = p + pre_s, p + post_s + 1
        if a < 0 or b > len(trace):
            continue
        snippets.append(trace[a:b])

    if len(snippets) == 0:
        return None, None, 0

    arr = np.vstack(snippets)
    mean = np.mean(arr, axis=0)
    sem = np.std(arr, axis=0, ddof=1) / np.sqrt(arr.shape[0]) if arr.shape[0] > 1 else np.zeros(arr.shape[1])
    rel_t = np.arange(pre_s, post_s + 1) * dt
    return rel_t, (mean, sem, arr), arr.shape[0]

def _window_mean(rel_t, arr2d, w):
    m = (rel_t >= float(w[0])) & (rel_t <= float(w[1]))
    if not np.any(m):
        return np.full(arr2d.shape[0], np.nan)
    return np.nanmean(arr2d[:, m], axis=1)

event_cache = {}
rows = []
all_snippets = []
snippets_by_cell = {i: [] for i in range(n_cells)}
rel_t_ref = None
skipped_for_pooling = 0

for s in processed_sessions:
    ev = _get_session_event_times_ms(s, EVENT_CONFIG, cache=event_cache)
    for c in range(s['n_cells']):
        # Use the spike-masked, low-pass subthreshold trace rather than the raw corrected trace.
        tr = s['subthreshold'][:, c]
        rel_t, out, n_ev = event_locked_subthreshold(tr, s['time_ms'], ev, window_ms=EVENT_LOCK_WINDOW_MS)
        if out is None:
            continue
        mean_curve, sem_curve, arr = out

        base_vals = _window_mean(rel_t, arr, EVENT_BASELINE_MS)
        resp_vals = _window_mean(rel_t, arr, EVENT_RESPONSE_MS)
        delta = resp_vals - base_vals

        rows.append({
            'session': s['session_name'],
            'cell': s['cell_names'][c],
            'n_events': int(n_ev),
            'subth_baseline_mean': float(np.nanmean(base_vals)),
            'subth_response_mean': float(np.nanmean(resp_vals)),
            'subth_delta_mean': float(np.nanmean(delta)),
        })

        if rel_t_ref is None:
            rel_t_ref = rel_t.copy()
        elif (len(rel_t) != len(rel_t_ref)) or (not np.allclose(rel_t, rel_t_ref, atol=1e-9)):
            skipped_for_pooling += 1
            continue

        all_snippets.append(arr)
        snippets_by_cell[c].append(arr)

event_subth_df = pd.DataFrame(rows)
display(event_subth_df.head(20))

if skipped_for_pooling > 0:
    print(f'Skipped {skipped_for_pooling} traces for pooling due to time-grid mismatch.')

# (a) Pooled across all cells and sessions
if rel_t_ref is not None and len(all_snippets) > 0:
    pooled_arr = np.vstack(all_snippets)
    pooled_mean = np.nanmean(pooled_arr, axis=0)
    pooled_sem = np.nanstd(pooled_arr, axis=0, ddof=1) / np.sqrt(pooled_arr.shape[0]) if pooled_arr.shape[0] > 1 else np.zeros(pooled_arr.shape[1])

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(rel_t_ref, pooled_mean, color=SUBTH_COLORS['pooled'], lw=2)
    ax.fill_between(rel_t_ref, pooled_mean - pooled_sem, pooled_mean + pooled_sem, color=SUBTH_COLORS['pooled'], alpha=0.2)
    ax.axvline(0, color='k', ls='--', lw=1)
    ax.set_title(f'Pooled event-locked subthreshold | all sessions + all cells | n_events={pooled_arr.shape[0]}')
    ax.set_xlabel('Time from event (ms)')
    ax.set_ylabel('Subthreshold signal (a.u.)')
    sns.despine(ax=ax, left=True, bottom=True)
    plt.show()

    pooled_base = _window_mean(rel_t_ref, pooled_arr, EVENT_BASELINE_MS)
    pooled_resp = _window_mean(rel_t_ref, pooled_arr, EVENT_RESPONSE_MS)
    pooled_delta = pooled_resp - pooled_base
    pooled_all_summary = pd.DataFrame([{
        'group': 'all_sessions_all_cells',
        'n_events': int(pooled_arr.shape[0]),
        'subth_baseline_mean': float(np.nanmean(pooled_base)),
        'subth_response_mean': float(np.nanmean(pooled_resp)),
        'subth_delta_mean': float(np.nanmean(pooled_delta)),
    }])
    display(pooled_all_summary)
else:
    print('No pooled event snippets available across all sessions/cells.')

# (b) Pooled by cell across all sessions
ncols = 2
nrows = int(np.ceil(n_cells / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.5 * nrows), sharex=True, sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

per_cell_rows = []
for c in range(n_cells):
    ax = axes[c]
    if len(snippets_by_cell[c]) == 0 or rel_t_ref is None:
        ax.set_title(f"{cell_names[c]} (no events)")
        sns.despine(ax=ax, left=True, bottom=True)
        continue

    cell_arr = np.vstack(snippets_by_cell[c])
    cell_mean = np.nanmean(cell_arr, axis=0)
    cell_sem = np.nanstd(cell_arr, axis=0, ddof=1) / np.sqrt(cell_arr.shape[0]) if cell_arr.shape[0] > 1 else np.zeros(cell_arr.shape[1])

    ax.plot(rel_t_ref, cell_mean, color=SUBTH_COLORS['ss'], lw=1.8)
    ax.fill_between(rel_t_ref, cell_mean - cell_sem, cell_mean + cell_sem, color=SUBTH_COLORS['ss'], alpha=0.2)
    ax.axvline(0, color='k', ls='--', lw=1)
    ax.set_title(f"{cell_names[c]} | pooled n_events={cell_arr.shape[0]}")
    sns.despine(ax=ax, left=True, bottom=True)

    base_vals = _window_mean(rel_t_ref, cell_arr, EVENT_BASELINE_MS)
    resp_vals = _window_mean(rel_t_ref, cell_arr, EVENT_RESPONSE_MS)
    delta = resp_vals - base_vals
    per_cell_rows.append({
        'cell': cell_names[c],
        'n_events': int(cell_arr.shape[0]),
        'subth_baseline_mean': float(np.nanmean(base_vals)),
        'subth_response_mean': float(np.nanmean(resp_vals)),
        'subth_delta_mean': float(np.nanmean(delta)),
    })

for i in range(n_cells, len(axes)):
    axes[i].axis('off')

fig.suptitle('', y=1.02)
axes[0].set_ylabel('Subthreshold signal (a.u.)')
for ax in axes[-ncols:]:
    ax.set_xlabel('Time from event (ms)')
plt.show()

per_cell_event_summary_df = pd.DataFrame(per_cell_rows)
display_top(per_cell_event_summary_df, label='Per-cell event summary')

In [ ]:
# ==============================
# Pairwise synchrony + whole-subthreshold feature report (focused)
# ==============================
PAIR_SYNC_CONFIG = {
    'min_window_samples': 25,
    'n_perm': 5000,
    'n_boot': 3000,
    'alpha': 0.05,
    'rng_seed': 123,
}


def _safe_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    good = np.isfinite(x) & np.isfinite(y)
    if np.sum(good) < 5:
        return np.nan
    x = x[good]
    y = y[good]
    sx = np.std(x)
    sy = np.std(y)
    if sx <= 0 or sy <= 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def _perm_p_two_sided(values, n_perm=5000, rng=None):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan
    obs = float(np.mean(vals))
    if np.allclose(vals, 0.0):
        return 1.0
    rng = np.random.default_rng() if rng is None else rng
    signs = rng.choice([-1.0, 1.0], size=(int(n_perm), len(vals)))
    null_means = np.mean(signs * vals[None, :], axis=1)
    p = (1.0 + np.sum(np.abs(null_means) >= abs(obs))) / (float(n_perm) + 1.0)
    return float(p)


def _bootstrap_ci_mean(values, n_boot=3000, alpha=0.05, rng=None):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng() if rng is None else rng
    idx = rng.integers(0, len(vals), size=(int(n_boot), len(vals)))
    boots = np.mean(vals[idx], axis=1)
    lo = float(np.quantile(boots, alpha / 2.0))
    hi = float(np.quantile(boots, 1.0 - alpha / 2.0))
    return lo, hi


def _bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    finite = np.isfinite(p)
    if not np.any(finite):
        return q
    pf = p[finite]
    m = len(pf)
    order = np.argsort(pf)
    ranked = pf[order]
    q_ranked = ranked * m / np.arange(1, m + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0.0, 1.0)
    back = np.empty_like(q_ranked)
    back[order] = q_ranked
    q[finite] = back
    return q


def _lag_autocorr(x, lag_samples):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) <= lag_samples + 5:
        return np.nan
    x0 = x[:-lag_samples]
    x1 = x[lag_samples:]
    return _safe_corr(x0, x1)


def _pair_matrix(cell_names, pair_summary, value_col):
    n = len(cell_names)
    mat = np.full((n, n), np.nan, dtype=float)
    np.fill_diagonal(mat, np.nan)
    for i, ci in enumerate(cell_names):
        for j, cj in enumerate(cell_names):
            if i >= j:
                continue
            m = pair_summary[(pair_summary['cell_i'] == ci) & (pair_summary['cell_j'] == cj)]
            if len(m) == 0:
                m = pair_summary[(pair_summary['cell_i'] == cj) & (pair_summary['cell_j'] == ci)]
            if len(m):
                v = float(m[value_col].iloc[0])
                mat[i, j] = v
                mat[j, i] = v
    return mat


def _pair_stat_matrix(cell_names, pair_summary, value_col):
    n = len(cell_names)
    mat = np.full((n, n), np.nan, dtype=float)
    np.fill_diagonal(mat, np.nan)
    for _, row in pair_summary.iterrows():
        ci = str(row['cell_i'])
        cj = str(row['cell_j'])
        if ci not in cell_names or cj not in cell_names:
            continue
        i = cell_names.index(ci)
        j = cell_names.index(cj)
        v = float(row[value_col]) if np.isfinite(row[value_col]) else np.nan
        mat[i, j] = v
        mat[j, i] = v
    return mat


def _pair_matrix_text_color(val, threshold=None):
    try:
        v = float(val)
    except Exception:
        return 'black'
    if not np.isfinite(v):
        return '#6B7280'
    if threshold is None:
        cfg = globals().get('CORR_HEATMAP_SCALE', {})
        if bool(cfg.get('enabled', False)):
            lim = max(abs(float(cfg.get('vmin', -1.0))), abs(float(cfg.get('vmax', 1.0))))
            threshold = 0.55 * lim
        else:
            threshold = 0.55
    return 'white' if abs(v) >= float(threshold) else 'black'


def _format_p(v):
    if not np.isfinite(v):
        return '=na'
    if v < 0.00001:
        return '<0.00001'
    return f'={float(v):.4f}'


def _p_stars(v):
    if not np.isfinite(v):
        return ''
    if v < 0.001:
        return '***'
    if v < 0.01:
        return '**'
    if v < 0.05:
        return '*'
    return ''


def _overlay_pair_matrix_text(ax, value_mat, p_mat=None, q_mat=None):
    value_fs = float(globals().get('SUBTH_CORR_MATRIX_STYLE', {}).get('value_fontsize', 16.9))
    stat_fs = float(globals().get('SUBTH_CORR_MATRIX_STYLE', {}).get('stat_fontsize', 11.8))
    for i in range(value_mat.shape[0]):
        for j in range(value_mat.shape[1]):
            val = value_mat[i, j]
            if not np.isfinite(val):
                continue
            x = j + 0.5
            y = i + 0.5
            txt_color = _pair_matrix_text_color(val)
            if i == j:
                ax.text(x, y, f'{val:.2f}', ha='center', va='center', fontsize=value_fs, fontweight='bold', color=txt_color)
                continue
            p_val = np.nan if p_mat is None else p_mat[i, j]
            q_val = np.nan if q_mat is None else q_mat[i, j]
            ax.text(x, y - 0.12, f"{val:.2f}{_p_stars(p_val)}", ha='center', va='center', fontsize=value_fs, fontweight='bold', color=txt_color)
            if p_mat is not None:
                ax.text(x, y + 0.10, f'p{_format_p(p_val)}', ha='center', va='center', fontsize=stat_fs, color=txt_color)
            if q_mat is not None and np.isfinite(q_val):
                ax.text(x, y + 0.24, f'q{_format_p(q_val)}', ha='center', va='center', fontsize=stat_fs, color=txt_color)


def _coherence_band_mean(x, y, fs, coh_cfg):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    good = np.isfinite(x) & np.isfinite(y)
    if np.sum(good) < 40:
        return np.nan

    x = x[good]
    y = y[good]
    n = x.size

    # Keep enough Welch segments in short event windows to avoid artificial MSC=1 saturation.
    nperseg_cfg = int(coh_cfg.get('nperseg', 512))
    nperseg = min(nperseg_cfg, max(32, n // 4))
    if nperseg >= n:
        nperseg = max(16, n // 2)
    if nperseg < 16:
        return np.nan

    noverlap_ratio = float(coh_cfg.get('noverlap_ratio', 0.5))
    noverlap = int(np.clip(noverlap_ratio * nperseg, 0, max(0, nperseg - 1)))

    f, msc = coherence(x, y, fs=float(fs), nperseg=nperseg, noverlap=noverlap)
    if msc.size == 0:
        return np.nan

    band = tuple(coh_cfg.get('freq_band_hz', (1.0, 50.0)))
    band_mask = (f >= float(band[0])) & (f <= float(band[1]))
    if not np.any(band_mask):
        return np.nan
    return float(np.nanmean(msc[band_mask]))


rng = np.random.default_rng(PAIR_SYNC_CONFIG['rng_seed'])
coh_rng = np.random.default_rng(int(COHERENCE_CONFIG.get('rng_seed', 123)))
matrix_tick_fs = float(globals().get('SUBTH_CORR_MATRIX_STYLE', {}).get('tick_label_fontsize', 15.2))
matrix_title_fs = float(globals().get('SUBTH_CORR_MATRIX_STYLE', {}).get('title_fontsize', 18.6))

if '_subthsync_event_times_for_session' not in globals():
    def _subthsync_event_times_for_session(session_obj):
        if '_get_session_event_times_ms' in globals():
            try:
                return np.asarray(_get_session_event_times_ms(session_obj, EVENT_CONFIG, cache={}), dtype=float)
            except Exception:
                pass
        return np.array([float(EVENT_ONSET_MS)], dtype=float)

if '_subthsync_mask_trace' not in globals():
    def _subthsync_mask_trace(trace, time_ms, event_times_ms, window_ms):
        out = np.asarray(trace, dtype=float).copy()
        t = np.asarray(time_ms, dtype=float)
        if out.size == 0 or t.size != out.size:
            return out
        w0, w1 = float(window_ms[0]), float(window_ms[1])
        bad = np.zeros(out.shape, dtype=bool)
        for event_time_ms in np.asarray(event_times_ms, dtype=float):
            bad |= (t >= (event_time_ms + w0)) & (t <= (event_time_ms + w1))
        out[bad] = np.nan
        return out

# --------------------------------------------------
# A) Event-locked pairwise synchrony (Pearson + MSC pre/post/delta + p/q)
# --------------------------------------------------
event_pair_rows = []
event_cache_pair = {}

for s in processed_sessions:
    t = np.asarray(s['time_ms'], dtype=float)
    tr = np.asarray(s['subthreshold'], dtype=float)
    fs = float(s['fs'])
    ev_ms = _get_session_event_times_ms(s, EVENT_CONFIG, cache=event_cache_pair)

    for ev in np.asarray(ev_ms, dtype=float):
        pre_mask = (t >= (ev + float(EVENT_BASELINE_MS[0]))) & (t <= (ev + float(EVENT_BASELINE_MS[1])))
        post_mask = (t >= (ev + float(EVENT_RESPONSE_MS[0]))) & (t <= (ev + float(EVENT_RESPONSE_MS[1])))
        if np.sum(pre_mask) < PAIR_SYNC_CONFIG['min_window_samples'] or np.sum(post_mask) < PAIR_SYNC_CONFIG['min_window_samples']:
            continue

        for i in range(s['n_cells']):
            for j in range(i + 1, s['n_cells']):
                x_pre = tr[pre_mask, i]
                y_pre = tr[pre_mask, j]
                x_post = tr[post_mask, i]
                y_post = tr[post_mask, j]

                r_pre = _safe_corr(x_pre, y_pre)
                r_post = _safe_corr(x_post, y_post)
                msc_pre = _coherence_band_mean(x_pre, y_pre, fs, COHERENCE_CONFIG)
                msc_post = _coherence_band_mean(x_post, y_post, fs, COHERENCE_CONFIG)

                if not (np.isfinite(r_pre) and np.isfinite(r_post)):
                    continue

                event_pair_rows.append({
                    'session': s['session_name'],
                    'event_ms': float(ev),
                    'cell_i': s['cell_names'][i],
                    'cell_j': s['cell_names'][j],
                    'pair': f"{s['cell_names'][i]} <-> {s['cell_names'][j]}",
                    'pre_r': float(r_pre),
                    'post_r': float(r_post),
                    'delta_r': float(r_post - r_pre),
                    'pre_msc': float(msc_pre) if np.isfinite(msc_pre) else np.nan,
                    'post_msc': float(msc_post) if np.isfinite(msc_post) else np.nan,
                    'delta_msc': float(msc_post - msc_pre) if (np.isfinite(msc_pre) and np.isfinite(msc_post)) else np.nan,
                })

event_pair_df = pd.DataFrame(event_pair_rows)
if len(event_pair_df) == 0:
    print('No valid event windows found for pre/post synchrony analysis.')
else:
    event_pair_summary = (
        event_pair_df
        .groupby(['cell_i', 'cell_j', 'pair'], as_index=False)
        .agg(
            n_event_windows=('delta_r', 'count'),
            pre_r_mean=('pre_r', 'mean'),
            post_r_mean=('post_r', 'mean'),
            delta_r_mean=('delta_r', 'mean'),
            delta_r_median=('delta_r', 'median'),
            pre_msc_mean=('pre_msc', 'mean'),
            post_msc_mean=('post_msc', 'mean'),
            delta_msc_mean=('delta_msc', 'mean'),
        )
    )

    robust_rows = []
    for _, row in event_pair_summary.iterrows():
        pair_name = row['pair']
        d_r = event_pair_df.loc[event_pair_df['pair'] == pair_name, 'delta_r'].values
        d_msc = event_pair_df.loc[event_pair_df['pair'] == pair_name, 'delta_msc'].values

        p_perm_r = _perm_p_two_sided(d_r, n_perm=PAIR_SYNC_CONFIG['n_perm'], rng=rng)
        ci_lo_r, ci_hi_r = _bootstrap_ci_mean(
            d_r,
            n_boot=PAIR_SYNC_CONFIG['n_boot'],
            alpha=PAIR_SYNC_CONFIG['alpha'],
            rng=rng,
        )
        p_perm_msc = _perm_p_two_sided(d_msc, n_perm=PAIR_SYNC_CONFIG['n_perm'], rng=rng)
        ci_lo_msc, ci_hi_msc = _bootstrap_ci_mean(
            d_msc,
            n_boot=PAIR_SYNC_CONFIG['n_boot'],
            alpha=PAIR_SYNC_CONFIG['alpha'],
            rng=rng,
        )

        robust_rows.append({
            'pair': pair_name,
            'perm_p_two_sided_r': p_perm_r,
            'delta_r_ci_lo': ci_lo_r,
            'delta_r_ci_hi': ci_hi_r,
            'delta_r_positive_fraction': float(np.mean(np.asarray(d_r) > 0)),
            'perm_p_two_sided_msc': p_perm_msc,
            'delta_msc_ci_lo': ci_lo_msc,
            'delta_msc_ci_hi': ci_hi_msc,
            'delta_msc_positive_fraction': float(np.mean(np.asarray(d_msc) > 0)) if np.any(np.isfinite(d_msc)) else np.nan,
        })

    robust_df = pd.DataFrame(robust_rows)
    robust_df['fdr_q_r'] = _bh_fdr(robust_df['perm_p_two_sided_r'].values)
    robust_df['fdr_q_msc'] = _bh_fdr(robust_df['perm_p_two_sided_msc'].values)
    event_pair_summary = event_pair_summary.merge(robust_df, on='pair', how='left')
    event_pair_summary = event_pair_summary.sort_values('delta_r_mean', ascending=False).reset_index(drop=True)

    delta_stats = event_pair_summary[
        [
            'pair', 'n_event_windows',
            'pre_r_mean', 'post_r_mean', 'delta_r_mean', 'perm_p_two_sided_r', 'fdr_q_r',
            'pre_msc_mean', 'post_msc_mean', 'delta_msc_mean', 'perm_p_two_sided_msc', 'fdr_q_msc',
        ]
    ].copy()
    display_top(delta_stats, label='Event-locked pair summary', sort_by='delta_r_mean')

    pre_r_mat = _pair_matrix(cell_names, event_pair_summary, 'pre_r_mean')
    post_r_mat = _pair_matrix(cell_names, event_pair_summary, 'post_r_mean')
    delta_r_mat = _pair_matrix(cell_names, event_pair_summary, 'delta_r_mean')
    p_r_mat = _pair_stat_matrix(cell_names, event_pair_summary, 'perm_p_two_sided_r')
    q_r_mat = _pair_stat_matrix(cell_names, event_pair_summary, 'fdr_q_r')

    pre_msc_mat = _pair_matrix(cell_names, event_pair_summary, 'pre_msc_mean')
    post_msc_mat = _pair_matrix(cell_names, event_pair_summary, 'post_msc_mean')
    delta_msc_mat = _pair_matrix(cell_names, event_pair_summary, 'delta_msc_mean')
    p_msc_mat = _pair_stat_matrix(cell_names, event_pair_summary, 'perm_p_two_sided_msc')
    q_msc_mat = _pair_stat_matrix(cell_names, event_pair_summary, 'fdr_q_msc')

    matrix_specs = [
        ('Pre-event pair correlation (r)', pre_r_mat, p_r_mat, q_r_mat, 'vlag', -1, 1, 'r', True),
        ('Post-event pair correlation (r)', post_r_mat, p_r_mat, q_r_mat, 'vlag', -1, 1, 'r', True),
        ('Delta pair correlation (post - pre)', delta_r_mat, p_r_mat, q_r_mat, 'vlag', -1, 1, 'delta r', True),
        ('Pre-event MSC (band mean)', pre_msc_mat, p_msc_mat, q_msc_mat, 'mako', 0, 1, 'MSC', False),
        ('Post-event MSC (band mean)', post_msc_mat, p_msc_mat, q_msc_mat, 'mako', 0, 1, 'MSC', False),
        ('Delta MSC (post - pre)', delta_msc_mat, p_msc_mat, q_msc_mat, 'mako', -0.5, 0.5, 'delta MSC', False),
    ]

    for matrix_title, matrix_values, matrix_p, matrix_q, cmap_name, vmin, vmax, cbar_label, use_center in matrix_specs:
        fig, ax = plt.subplots(1, 1, figsize=(7.3, 5.6), constrained_layout=True)
        heatmap_kwargs = dict(
            data=matrix_values,
            annot=False,
            fmt='',
            cmap=cmap_name,
            vmin=vmin,
            vmax=vmax,
            square=True,
            xticklabels=cell_names,
            yticklabels=cell_names,
            cbar_kws={'label': cbar_label},
            linewidths=0.5,
            linecolor='white',
            ax=ax,
        )
        if use_center:
            heatmap_kwargs['center'] = 0
        sns.heatmap(**heatmap_kwargs)
        _overlay_pair_matrix_text(ax, matrix_values, matrix_p, matrix_q)
        ax.set_title(matrix_title, fontsize=matrix_title_fs)
        ax.tick_params(axis='x', rotation=25, labelsize=matrix_tick_fs)
        ax.tick_params(axis='y', rotation=0, labelsize=matrix_tick_fs)
        if getattr(ax, 'collections', None):
            cbar = ax.collections[0].colorbar
            if cbar is not None:
                cbar.ax.tick_params(labelsize=float(SUBTH_CORR_MATRIX_STYLE.get('colorbar_tick_fontsize', 8)))
                cbar.set_label(cbar.ax.get_ylabel(), size=float(SUBTH_CORR_MATRIX_STYLE.get('colorbar_label_fontsize', 9)))
        sns.despine(ax=ax, left=True, bottom=True)
        plt.show()

# --------------------------------------------------
# B) Non-event-locked synchrony robustness (whole recording)
# --------------------------------------------------
global_rows = []
mask_post_event_period = bool(SUBTH_SYNC_CONFIG.get('mask_post_event_period', False))
if mask_post_event_period:
    print(f'Whole-recording subthreshold pair summary masks EVENT_RESPONSE_MS={EVENT_RESPONSE_MS} around each event before correlation.')

for s in processed_sessions:
    raw_tr = np.asarray(s['subthreshold'], dtype=float)
    if mask_post_event_period:
        event_times_ms = _subthsync_event_times_for_session(s)
        tr = np.column_stack([
            _subthsync_mask_trace(raw_tr[:, cell_idx], s['time_ms'], event_times_ms, EVENT_RESPONSE_MS)
            for cell_idx in range(s['n_cells'])
        ])
    else:
        tr = raw_tr
    for i in range(s['n_cells']):
        for j in range(i + 1, s['n_cells']):
            r0 = _safe_corr(tr[:, i], tr[:, j])
            msc0 = _coherence_band_mean(tr[:, i], tr[:, j], float(s['fs']), COHERENCE_CONFIG)
            global_rows.append({
                'session': s['session_name'],
                'cell_i': s['cell_names'][i],
                'cell_j': s['cell_names'][j],
                'pair': f"{s['cell_names'][i]} <-> {s['cell_names'][j]}",
                'whole_recording_r': r0,
                'whole_recording_msc': msc0,
            })

global_pair_df = pd.DataFrame(global_rows)

global_pair_summary = (
    global_pair_df
    .groupby(['cell_i', 'cell_j', 'pair'], as_index=False)
    .agg(
        n_sessions=('whole_recording_r', 'count'),
        mean_r=('whole_recording_r', 'mean'),
        median_r=('whole_recording_r', 'median'),
        std_r=('whole_recording_r', 'std'),
        mean_msc=('whole_recording_msc', 'mean'),
        median_msc=('whole_recording_msc', 'median'),
        std_msc=('whole_recording_msc', 'std'),
    )
    .reset_index(drop=True)
)

global_robust_rows = []
for _, row in global_pair_summary.iterrows():
    pair_name = row['pair']
    vals = global_pair_df.loc[global_pair_df['pair'] == pair_name, 'whole_recording_r'].values
    msc_vals = global_pair_df.loc[global_pair_df['pair'] == pair_name, 'whole_recording_msc'].values
    p_perm = _perm_p_two_sided(vals, n_perm=PAIR_SYNC_CONFIG['n_perm'], rng=rng)
    ci_lo, ci_hi = _bootstrap_ci_mean(
        vals,
        n_boot=PAIR_SYNC_CONFIG['n_boot'],
        alpha=PAIR_SYNC_CONFIG['alpha'],
        rng=rng,
    )
    p_perm_msc = _perm_p_two_sided(msc_vals, n_perm=PAIR_SYNC_CONFIG['n_perm'], rng=rng)
    ci_lo_msc, ci_hi_msc = _bootstrap_ci_mean(
        msc_vals,
        n_boot=PAIR_SYNC_CONFIG['n_boot'],
        alpha=PAIR_SYNC_CONFIG['alpha'],
        rng=rng,
    )
    global_robust_rows.append({
        'pair': pair_name,
        'perm_p_two_sided': p_perm,
        'mean_r_ci_lo': ci_lo,
        'mean_r_ci_hi': ci_hi,
        'positive_fraction': float(np.mean(np.asarray(vals) > 0)),
        'perm_p_two_sided_msc': p_perm_msc,
        'mean_msc_ci_lo': ci_lo_msc,
        'mean_msc_ci_hi': ci_hi_msc,
    })

global_robust_df = pd.DataFrame(global_robust_rows)
global_robust_df['fdr_q'] = _bh_fdr(global_robust_df['perm_p_two_sided'].values)
global_robust_df['fdr_q_msc'] = _bh_fdr(global_robust_df['perm_p_two_sided_msc'].values)
global_pair_summary = global_pair_summary.merge(global_robust_df, on='pair', how='left')
global_pair_summary = global_pair_summary.sort_values('mean_r', ascending=False).reset_index(drop=True)
display_top(global_pair_summary, label='Whole-recording pair summary')

print('Method detail (non-event-locked robustness):')
print('1) Compute whole-recording zero-lag Pearson r and MSC-band mean for each pair in each session.')
print('2) For each pair, test whether mean metric differs from 0 with a two-sided sign-flip permutation test.')
print('3) Build bootstrap confidence intervals for mean metric by resampling session-level values with replacement.')
print('4) Correct pairwise p-values using Benjamini-Hochberg FDR to obtain q-values.')
print('5) Report robust synchrony as: mean metric, CI, permutation p, and FDR q.')

# --------------------------------------------------
# C) Whole subthreshold feature report (not only synchrony)
# --------------------------------------------------
feature_rows = []
for s in processed_sessions:
    tr = np.asarray(s['subthreshold'], dtype=float)
    fs = float(s['fs'])
    t = np.asarray(s['time_ms'], dtype=float)
    duration_s = float((t[-1] - t[0]) / 1000.0) if len(t) > 1 else np.nan
    lag_100 = max(1, int(round(0.100 * fs)))

    for c, cname in enumerate(s['cell_names']):
        x = np.asarray(tr[:, c], dtype=float)
        gx = x[np.isfinite(x)]
        if len(gx) < 5:
            continue
        dx = np.diff(gx)
        feature_rows.append({
            'session': s['session_name'],
            'cell_name': cname,
            'duration_s': duration_s,
            'subth_mean': float(np.mean(gx)),
            'subth_std': float(np.std(gx)),
            'subth_iqr': float(np.quantile(gx, 0.75) - np.quantile(gx, 0.25)),
            'subth_rms': float(np.sqrt(np.mean(gx**2))),
            'subth_range': float(np.max(gx) - np.min(gx)),
            'subth_mad': float(np.median(np.abs(gx - np.median(gx)))),
            'roughness_mean_abs_diff': float(np.mean(np.abs(dx))) if len(dx) else np.nan,
            'autocorr_100ms': _lag_autocorr(gx, lag_100),
        })

subth_feature_df = pd.DataFrame(feature_rows)

subth_feature_summary = (
    subth_feature_df
    .groupby('cell_name', as_index=False)
    .agg(
        n_session_cells=('session', 'count'),
        duration_s_mean=('duration_s', 'mean'),
        mean_of_mean=('subth_mean', 'mean'),
        mean_of_std=('subth_std', 'mean'),
        mean_of_iqr=('subth_iqr', 'mean'),
        mean_of_rms=('subth_rms', 'mean'),
        mean_of_range=('subth_range', 'mean'),
        mean_of_mad=('subth_mad', 'mean'),
        mean_of_roughness=('roughness_mean_abs_diff', 'mean'),
        mean_autocorr_100ms=('autocorr_100ms', 'mean'),
    )
    .sort_values('mean_of_std', ascending=False)
    .reset_index(drop=True)
)
display_top(subth_feature_summary, label='Subthreshold feature summary')

print('Whole subthreshold dataset summary:')
print(f"- Sessions analyzed: {len(processed_sessions)}")
print(f"- Cells per session: {n_cells}")
print(f"- Session-cell traces summarized: {len(subth_feature_df)}")
if len(global_pair_df) > 0:
    valid_global = global_pair_df['whole_recording_r'].dropna().values
    print(f"- Whole-recording synchrony mean r: {np.mean(valid_global):.3f}, median r: {np.median(valid_global):.3f}")
    print(f"- Whole-recording synchrony positive fraction: {np.mean(valid_global > 0) * 100:.1f}%")

## Notes

- To use manual event timing only, set `EVENT_DETECTION_CONFIG['auto_detect_event'] = False`.
- To compare filter choices, rerun with `LOWPASS_CUTOFF_HZ` = 50, 80, and 100.
- To increase spike artifact suppression, widen `SPIKE_MASK_SS_MS` / `SPIKE_MASK_CS_MS`.
- If `EXPORT_RESULTS=True`, you can extend export by saving `spike_subth_df` and `event_subth_df` to CSV in `EXPORT_DIR`.